In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
OUTPUT_DIR = Path('RSV_PUBLICATION_ANALYSIS')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PARAMETERS = ['alpha', 'season_amp', 'sigma', 'shift_weeks']
RHAT_GOOD = 1.01
RHAT_CAUTION = 1.05
FIT_FILE = Path('RSV_Fit_Results.csv')
POSTERIOR_FILE = Path('RSV_Posterior_Samples.dat')
POSTERIOR_PERFORMANCE_FILE = Path('RSV_Posterior_Corr_Dist.dat')
DIAGNOSTICS_FILE = Path('RSV_MCMC_Diagnostics.dat')
MLE_FILE = Path('RSV_MLE_Summary.dat')
LOO_FILE = Path('RSV_LOO_Predictions.dat')
PARAMETER_SUMMARY_FILE = Path('RSV_Parameter_Summary.dat')
plt.rcParams.update({'font.family': 'serif', 'font.size': 12, 'axes.spines.top': True, 'axes.spines.right': True, 'axes.spines.bottom': True, 'axes.spines.left': True, 'axes.linewidth': 1.1, 'figure.dpi': 150, 'savefig.dpi': 300})

def require_file(path):
    if not path.exists():
        raise FileNotFoundError(f'\nRequired file not found:\n  {path.resolve()}')

def optional_file(path):
    if not path.exists():
        print(f'WARNING: File not found: {path}')
        return False
    return True

def to_float(x):
    return float(str(x).replace('D', 'E').replace('d', 'e'))

def q95(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return (np.nan, np.nan, np.nan)
    return tuple(np.percentile(x, [2.5, 50.0, 97.5]))

def pearson_r(y, f):
    y = np.asarray(y, dtype=float)
    f = np.asarray(f, dtype=float)
    m = np.isfinite(y) & np.isfinite(f)
    if np.sum(m) < 2:
        return np.nan
    if np.std(y[m]) == 0 or np.std(f[m]) == 0:
        return np.nan
    return float(np.corrcoef(y[m], f[m])[0, 1])

def rmse(y, f):
    y = np.asarray(y, dtype=float)
    f = np.asarray(f, dtype=float)
    m = np.isfinite(y) & np.isfinite(f)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((y[m] - f[m]) ** 2)))

def read_numeric(path):
    """
    Read numerical whitespace-separated rows from a Fortran output file.
    Text headers and comments are ignored.
    """
    rows = []
    with open(path, 'r', errors='ignore') as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith('#'):
                continue
            parts = s.split()
            try:
                row = [to_float(x) for x in parts]
            except (ValueError, TypeError):
                continue
            rows.append(row)
    if not rows:
        return None
    lengths = {}
    for row in rows:
        lengths[len(row)] = lengths.get(len(row), 0) + 1
    ncol = max(lengths, key=lengths.get)
    rows = [r for r in rows if len(r) == ncol]
    return np.asarray(rows, dtype=float)

def savefig(name):
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / name, dpi=300, bbox_inches='tight')
    plt.close()

def load_fit():
    require_file(FIT_FILE)
    df = pd.read_csv(FIT_FILE)
    df.columns = df.columns.str.strip()
    print('\nFIT FILE')
    print('-' * 80)
    print(f'File: {FIT_FILE}')
    print(f'Columns: {list(df.columns)}')
    aliases = {'week': ['week'], 'r_data': ['r_data'], 'mle_curve': ['mle_curve'], 'posterior_median': ['posterior_median', 'post_median'], 'posterior_lower': ['posterior_lower', 'post_lower'], 'posterior_upper': ['posterior_upper', 'post_upper']}
    selected = {}
    for standard_name, possible_names in aliases.items():
        found = None
        for name in possible_names:
            if name in df.columns:
                found = name
                break
        if found is None:
            raise ValueError(f"\n{FIT_FILE} is missing the required column for '{standard_name}'.\n\nAccepted names: {possible_names}\n\nAvailable columns: {list(df.columns)}")
        selected[standard_name] = found
    result = {'week': df[selected['week']].to_numpy(float), 'r_data': df[selected['r_data']].to_numpy(float), 'mle_curve': df[selected['mle_curve']].to_numpy(float), 'posterior_median': df[selected['posterior_median']].to_numpy(float), 'posterior_lower': df[selected['posterior_lower']].to_numpy(float), 'posterior_upper': df[selected['posterior_upper']].to_numpy(float)}
    if 'loo_prediction' in df.columns:
        result['loo_prediction'] = df['loo_prediction'].to_numpy(float)
    if 'loo_error' in df.columns:
        result['loo_error'] = df['loo_error'].to_numpy(float)
    if 'loo_success' in df.columns:
        result['loo_success'] = df['loo_success'].to_numpy(float)
    print(f"\nSuccessfully loaded {len(result['week'])} observations.")
    print('\nColumn mapping:')
    for standard, actual in selected.items():
        print(f'  {standard:<20} <- {actual}')
    return result

def load_posterior():
    if not optional_file(POSTERIOR_FILE):
        return None
    data = read_numeric(POSTERIOR_FILE)
    if data is None:
        print('WARNING: Posterior file contains no numerical samples.')
        return None
    if data.shape[1] < 4:
        raise ValueError(f'{POSTERIOR_FILE} has {data.shape[1]} columns; at least 4 parameter columns are required.')
    print('\nPOSTERIOR PARAMETER SAMPLES')
    print('-' * 80)
    print(f'Samples: {data.shape[0]}')
    print(f'Columns used: alpha, season_amp, sigma, shift_weeks')
    return data[:, :4]

def load_posterior_performance():
    if not optional_file(POSTERIOR_PERFORMANCE_FILE):
        return None
    data = read_numeric(POSTERIOR_PERFORMANCE_FILE)
    if data is None:
        return None
    if data.shape[1] < 2:
        raise ValueError(f'{POSTERIOR_PERFORMANCE_FILE} needs at least two numerical columns.')
    print('\nPOSTERIOR PERFORMANCE')
    print('-' * 80)
    print(f'Rows: {data.shape[0]}')
    return {'correlation': data[:, 0], 'rmse': data[:, 1]}

def load_mle():
    if not optional_file(MLE_FILE):
        return None
    text = MLE_FILE.read_text(errors='ignore')
    result = {}
    patterns = {'alpha': '(?im)^\\s*alpha\\s*[:=]\\s*([-+0-9.eEdD]+)', 'season_amp': '(?im)^\\s*season[_ ]?amp(?:litude)?\\s*[:=]\\s*([-+0-9.eEdD]+)', 'sigma': '(?im)^\\s*sigma\\s*[:=]\\s*([-+0-9.eEdD]+)', 'shift_weeks': '(?im)^\\s*shift[_ ]?weeks\\s*[:=]\\s*([-+0-9.eEdD]+)', 'logLik': '(?im)^\\s*(?:log[_ ]?likelihood|loglik)\\s*[:=]\\s*([-+0-9.eEdD]+)'}
    for key, pattern in patterns.items():
        match = re.search(pattern, text)
        if match:
            result[key] = to_float(match.group(1))
    print('\nMLE SUMMARY')
    print('-' * 80)
    print(f'File: {MLE_FILE}')
    if result:
        for k, v in result.items():
            print(f'{k:<20}: {v:.12g}')
    else:
        print('WARNING: Could not identify named MLE values.')
    return result

def load_parameter_summary():
    if not optional_file(PARAMETER_SUMMARY_FILE):
        return None
    data = read_numeric(PARAMETER_SUMMARY_FILE)
    if data is None:
        return None
    print('\nPARAMETER SUMMARY')
    print('-' * 80)
    print(f'File: {PARAMETER_SUMMARY_FILE}')
    return data

def load_diagnostics():
    if not optional_file(DIAGNOSTICS_FILE):
        return None
    lines = DIAGNOSTICS_FILE.read_text(errors='ignore').splitlines()
    acceptance = []
    rhat = None
    for line in lines:
        s = line.strip()
        if not s:
            continue
        if s.startswith('#'):
            continue
        parts = s.replace(':', ' ').split()
        if len(parts) == 2:
            try:
                first = int(float(parts[0]))
                second = to_float(parts[1])
                if 1 <= first <= 4:
                    acceptance.append((first, second))
                    continue
            except (ValueError, TypeError):
                pass
        if len(parts) == 4:
            try:
                values = [to_float(x) for x in parts]
                if np.all(np.isfinite(values)):
                    rhat = np.asarray(values, dtype=float)
            except (ValueError, TypeError):
                pass
    overall = np.nan
    if acceptance:
        overall = float(np.mean([x[1] for x in acceptance]))
    max_rhat = np.nan
    if rhat is not None:
        max_rhat = float(np.max(rhat))
    if np.isfinite(max_rhat):
        if max_rhat <= RHAT_GOOD:
            convergence = 'GOOD'
        elif max_rhat <= RHAT_CAUTION:
            convergence = 'CAUTION'
        else:
            convergence = 'NOT CONVERGED'
    else:
        convergence = 'UNAVAILABLE'
    print('\nMCMC DIAGNOSTICS')
    print('-' * 80)
    print(f'File: {DIAGNOSTICS_FILE}')
    if acceptance:
        for chain, acc in acceptance:
            print(f'Chain {chain} acceptance: {acc:.10f}')
        print(f'Mean chain acceptance: {overall:.10f}')
    else:
        print('Acceptance values unavailable.')
    if rhat is not None:
        for p, value in zip(PARAMETERS, rhat):
            print(f'R-hat {p:<15}: {value:.10f}')
        print(f'Maximum R-hat:      {max_rhat:.10f}')
        print(f'Convergence:        {convergence}')
    else:
        print('Split-Rhat unavailable.')
    return {'acceptance': acceptance, 'overall_acceptance': overall, 'rhat': rhat, 'max_rhat': max_rhat, 'convergence': convergence}

def load_loo():
    if not optional_file(LOO_FILE):
        return None
    data = read_numeric(LOO_FILE)
    if data is None:
        return None
    print('\nLOO PREDICTIONS')
    print('-' * 80)
    print(f'File: {LOO_FILE}')
    print(f'Rows: {data.shape[0]}')
    print(f'Columns: {data.shape[1]}')
    return data

def posterior_parameter_statistics(samples):
    if samples is None:
        return None
    rows = []
    for j, p in enumerate(PARAMETERS):
        lower, median, upper = q95(samples[:, j])
        rows.append({'parameter': p, 'median': median, 'lower_95_CrI': lower, 'upper_95_CrI': upper, 'mean': np.mean(samples[:, j]), 'sd': np.std(samples[:, j], ddof=1)})
    return pd.DataFrame(rows)

def trajectory_metrics(fit):
    obs = fit['r_data']
    return {'MLE Pearson r': pearson_r(obs, fit['mle_curve']), 'MLE RMSE': rmse(obs, fit['mle_curve']), 'Posterior median Pearson r': pearson_r(obs, fit['posterior_median']), 'Posterior median RMSE': rmse(obs, fit['posterior_median'])}

def calculate_coverage(fit):
    obs = fit['r_data']
    lower = fit['posterior_lower']
    upper = fit['posterior_upper']
    inside = (obs >= lower) & (obs <= upper)
    n_inside = int(np.sum(inside))
    coverage = 100.0 * n_inside / len(obs)
    return (n_inside, len(obs), coverage, inside)

def calculate_loo_metrics(fit, loo):
    if loo is None:
        return None
    obs = fit['r_data']
    if 'loo_prediction' in fit:
        pred = fit['loo_prediction']
        success = np.ones(len(pred), dtype=bool)
        if 'loo_success' in fit:
            success = fit['loo_success'] > 0.5
        valid = success & np.isfinite(obs) & np.isfinite(pred)
        if np.sum(valid) >= 2:
            return {'n': int(np.sum(valid)), 'pearson_r': pearson_r(obs[valid], pred[valid]), 'rmse': rmse(obs[valid], pred[valid])}
    if len(loo) == len(obs) and loo.shape[1] >= 2:
        a = loo[:, 0]
        b = loo[:, 1]
        if np.nanmean(np.abs(a - obs)) <= np.nanmean(np.abs(b - obs)):
            pred = b
        else:
            pred = a
        valid = np.isfinite(obs) & np.isfinite(pred)
        if np.sum(valid) >= 2:
            return {'n': int(np.sum(valid)), 'pearson_r': pearson_r(obs[valid], pred[valid]), 'rmse': rmse(obs[valid], pred[valid])}
    return None

def print_results(fit, mle, posterior, post_perf, diagnostics, loo):
    metrics = trajectory_metrics(fit)
    coverage = calculate_coverage(fit)
    loo_metrics = calculate_loo_metrics(fit, loo)
    print('\n')
    print('=' * 100)
    print('RSV LYON - PUBLICATION RESULTS')
    print('=' * 100)
    print('\nDATA')
    print('-' * 100)
    print(f"Number of observations: {len(fit['r_data'])}")
    print('\n1. MLE RESULTS')
    print('-' * 100)
    if mle is not None:
        for p in PARAMETERS:
            if p in mle:
                print(f'{p:<22}: {mle[p]:.12g}')
        if 'logLik' in mle:
            print(f"{'Log-likelihood':<22}: {mle['logLik']:.12g}")
    else:
        print('MLE parameter summary unavailable.')
    print(f"{'Pearson r':<22}: {metrics['MLE Pearson r']:.10f}")
    print(f"{'RMSE':<22}: {metrics['MLE RMSE']:.10f}")
    print('\n2. POSTERIOR PARAMETERS — 95% CrI')
    print('-' * 100)
    pstats = posterior_parameter_statistics(posterior)
    if pstats is None:
        print('Posterior parameter samples unavailable.')
    else:
        print(f"{'Parameter':<18}{'Median':>18}{'2.5%':>18}{'97.5%':>18}")
        for _, row in pstats.iterrows():
            print(f"{row['parameter']:<18}{row['median']:>18.10f}{row['lower_95_CrI']:>18.10f}{row['upper_95_CrI']:>18.10f}")
    print('\n3. POSTERIOR MEDIAN TRAJECTORY')
    print('-' * 100)
    print(f"Pearson r:              {metrics['Posterior median Pearson r']:.10f}")
    print(f"RMSE:                   {metrics['Posterior median RMSE']:.10f}")
    print('\n4. POSTERIOR PERFORMANCE DISTRIBUTION')
    print('-' * 100)
    if post_perf is None:
        print('Posterior performance distribution unavailable.')
    else:
        rlo, rmed, rhi = q95(post_perf['correlation'])
        elo, emed, ehi = q95(post_perf['rmse'])
        print(f'Pearson r median:       {rmed:.10f}')
        print(f'Pearson r 95% CrI:      [{rlo:.10f}, {rhi:.10f}]')
        print(f'RMSE median:            {emed:.10f}')
        print(f'RMSE 95% CrI:           [{elo:.10f}, {ehi:.10f}]')
    print('-' * 100)
    print('\n6. 95% LATENT CrI COVERAGE')
    print('-' * 100)
    n_inside, n_total, cov, _ = coverage
    print(f'Observed points inside: {n_inside}/{n_total}')
    print(f'Coverage:               {cov:.2f}%')
    print('NOTE: This is latent trajectory coverage, not full predictive coverage.')
    print('\n7. MCMC DIAGNOSTICS')
    print('-' * 100)
    if diagnostics is None:
        print('MCMC diagnostics unavailable.')
    else:
        if diagnostics['acceptance']:
            for chain, acc in diagnostics['acceptance']:
                print(f'Chain {chain} acceptance: {acc:.10f}')
            print(f"Mean chain acceptance: {diagnostics['overall_acceptance']:.10f}")
        if diagnostics['rhat'] is not None:
            print('\nSplit-Rhat:')
            for p, value in zip(PARAMETERS, diagnostics['rhat']):
                print(f'  {p:<18}: {value:.10f}')
            print(f"\nMaximum Rhat: {diagnostics['max_rhat']:.10f}")
            print(f"Convergence assessment: {diagnostics['convergence']}")
    print('\n8. LOO PREDICTIVE PERFORMANCE')
    print('-' * 100)
    if loo_metrics is None:
        print('LOO metrics unavailable.')
    else:
        print(f"Valid LOO predictions:  {loo_metrics['n']}")
        print(f"LOO Pearson r:          {loo_metrics['pearson_r']:.10f}")
        print(f"LOO RMSE:               {loo_metrics['rmse']:.10f}")
    print('\n' + '=' * 100)

def plot_main_fit(fit):
    week = fit['week']
    obs = fit['r_data']
    mle = fit['mle_curve']
    median = fit['posterior_median']
    lower = fit['posterior_lower']
    upper = fit['posterior_upper']
    plt.figure(figsize=(10, 6))
    plt.fill_between(week, lower, upper, alpha=0.25, label='95% posterior CrI')
    plt.plot(week, obs, 'o', markersize=5, label='Observed')
    plt.plot(week, mle, '-', linewidth=2, label='MLE')
    plt.plot(week, median, '--', linewidth=2, label='Posterior median')
    if 'loo_prediction' in fit:
        loo = fit['loo_prediction']
        if 'loo_success' in fit:
            ok = fit['loo_success'] > 0.5
        else:
            ok = np.isfinite(loo)
        if np.any(ok):
            plt.plot(week[ok], loo[ok], '^', markersize=4, label='LOO prediction')
    plt.xlabel('Week')
    plt.ylabel('$R_{\\mathrm{eff}}$')
    plt.title('RSV Lyon: observed and fitted trajectories')
    plt.legend(frameon=False)
    savefig('01_RSV_observed_MLE_posterior_95CrI.png')

def plot_parameter_posteriors(posterior):
    if posterior is None:
        return
    fig, axes = plt.subplots(2, 2, figsize=(10, 7))
    for j, p in enumerate(PARAMETERS):
        ax = axes.ravel()[j]
        x = posterior[:, j]
        ax.hist(x, bins=40, density=True, alpha=0.75)
        lo, med, hi = q95(x)
        ax.axvline(med, linestyle='--', linewidth=1.5)
        ax.set_xlabel(p)
        ax.set_ylabel('Density')
        ax.set_title(f'Median = {med:.5g}\n95% CrI = [{lo:.5g}, {hi:.5g}]')
    fig.suptitle('RSV posterior parameter distributions')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / '02_RSV_posterior_parameters.png', dpi=300, bbox_inches='tight')
    plt.close(fig)

def plot_posterior_performance(post_perf):
    if post_perf is None:
        return
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    rlo, rmed, rhi = q95(post_perf['correlation'])
    axes[0].hist(post_perf['correlation'], bins=40, density=True, alpha=0.75)
    axes[0].axvline(rmed, linestyle='--', linewidth=1.5)
    axes[0].set_xlabel('Pearson r')
    axes[0].set_ylabel('Density')
    axes[0].set_title(f'Pearson r\nMedian={rmed:.4f}\n95% CrI=[{rlo:.4f}, {rhi:.4f}]')
    elo, emed, ehi = q95(post_perf['rmse'])
    axes[1].hist(post_perf['rmse'], bins=40, density=True, alpha=0.75)
    axes[1].axvline(emed, linestyle='--', linewidth=1.5)
    axes[1].set_xlabel('RMSE')
    axes[1].set_ylabel('Density')
    axes[1].set_title(f'RMSE\nMedian={emed:.4f}\n95% CrI=[{elo:.4f}, {ehi:.4f}]')
    fig.suptitle('RSV posterior model performance')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / '03_RSV_posterior_performance.png', dpi=300, bbox_inches='tight')
    plt.close(fig)

def plot_acceptance(diagnostics):
    if diagnostics is None:
        return
    if not diagnostics['acceptance']:
        return
    chains = np.asarray([x[0] for x in diagnostics['acceptance']])
    values = np.asarray([x[1] for x in diagnostics['acceptance']])
    plt.figure(figsize=(8, 5))
    plt.bar(chains, values)
    plt.axhline(diagnostics['overall_acceptance'], linestyle='--', linewidth=1.5, label=f"Mean = {diagnostics['overall_acceptance']:.4f}")
    plt.xlabel('MCMC chain')
    plt.ylabel('Acceptance fraction')
    plt.title('RSV MCMC chain acceptance')
    plt.xticks(chains)
    plt.legend(frameon=False)
    savefig('04_RSV_MCMC_acceptance.png')

def plot_rhat(diagnostics):
    if diagnostics is None:
        return
    if diagnostics['rhat'] is None:
        return
    x = np.arange(len(PARAMETERS))
    plt.figure(figsize=(8, 5))
    plt.bar(x, diagnostics['rhat'])
    plt.axhline(RHAT_GOOD, linestyle='--', linewidth=1.5, label='R-hat = 1.01')
    plt.axhline(RHAT_CAUTION, linestyle=':', linewidth=1.5, label='R-hat = 1.05')
    plt.xticks(x, PARAMETERS)
    plt.ylabel('Split-Rhat')
    plt.title('RSV MCMC convergence')
    plt.legend(frameon=False)
    savefig('05_RSV_Rhat.png')

def plot_residuals(fit):
    week = fit['week']
    obs = fit['r_data']
    plt.figure(figsize=(10, 5.5))
    plt.axhline(0, linestyle='--', linewidth=1)
    plt.plot(week, obs - fit['mle_curve'], 'o-', label='MLE residual')
    plt.plot(week, obs - fit['posterior_median'], 's--', label='Posterior median residual')
    plt.xlabel('Week')
    plt.ylabel('Observed - fitted')
    plt.title('RSV residuals')
    plt.legend(frameon=False)
    savefig('06_RSV_residuals.png')

def plot_observed_vs_fitted(fit):
    obs = fit['r_data']
    values = np.concatenate([obs, fit['mle_curve'], fit['posterior_median']])
    values = values[np.isfinite(values)]
    vmin = np.min(values)
    vmax = np.max(values)
    pad = 0.05 * max(vmax - vmin, 1.0)
    plt.figure(figsize=(6.5, 6))
    plt.plot([vmin - pad, vmax + pad], [vmin - pad, vmax + pad], '--', linewidth=1.2, label='1:1')
    plt.scatter(obs, fit['mle_curve'], s=40, label='MLE')
    plt.scatter(obs, fit['posterior_median'], s=40, marker='s', label='Posterior median')
    plt.xlabel('Observed $R_{eff}$')
    plt.ylabel('Fitted $R_{eff}$')
    plt.title('RSV observed versus fitted')
    plt.legend(frameon=False)
    savefig('07_RSV_observed_vs_fitted.png')

def plot_loo(fit):
    if 'loo_prediction' not in fit:
        return
    week = fit['week']
    obs = fit['r_data']
    pred = fit['loo_prediction']
    if 'loo_success' in fit:
        ok = fit['loo_success'] > 0.5
    else:
        ok = np.isfinite(pred)
    if not np.any(ok):
        return
    plt.figure(figsize=(10, 6))
    plt.plot(week, obs, 'o', markersize=5, label='Observed')
    plt.plot(week[ok], pred[ok], 's--', linewidth=1.7, markersize=4, label='LOO prediction')
    plt.xlabel('Week')
    plt.ylabel('$R_{\\mathrm{eff}}$')
    plt.title('RSV leave-one-out predictions')
    plt.legend(frameon=False)
    savefig('08_RSV_LOO_predictions.png')

def save_summary(fit, mle, posterior, post_perf, diagnostics, loo):
    rows = []

    def add(metric, value):
        rows.append({'metric': metric, 'value': value})
    add('n_observations', len(fit['r_data']))
    if mle is not None:
        for p in PARAMETERS:
            if p in mle:
                add(f'MLE_{p}', mle[p])
        if 'logLik' in mle:
            add('MLE_log_likelihood', mle['logLik'])
    add('MLE_Pearson_r', pearson_r(fit['r_data'], fit['mle_curve']))
    add('MLE_RMSE', rmse(fit['r_data'], fit['mle_curve']))
    add('Posterior_median_Pearson_r', pearson_r(fit['r_data'], fit['posterior_median']))
    add('Posterior_median_RMSE', rmse(fit['r_data'], fit['posterior_median']))
    pstats = posterior_parameter_statistics(posterior)
    if pstats is not None:
        for _, row in pstats.iterrows():
            p = row['parameter']
            add(f'Posterior_{p}_median', row['median'])
            add(f'Posterior_{p}_95CrI_lower', row['lower_95_CrI'])
            add(f'Posterior_{p}_95CrI_upper', row['upper_95_CrI'])
    if post_perf is not None:
        lo, med, hi = q95(post_perf['correlation'])
        add('Posterior_Pearson_r_median', med)
        add('Posterior_Pearson_r_95CrI_lower', lo)
        add('Posterior_Pearson_r_95CrI_upper', hi)
        lo, med, hi = q95(post_perf['rmse'])
        add('Posterior_RMSE_median', med)
        add('Posterior_RMSE_95CrI_lower', lo)
        add('Posterior_RMSE_95CrI_upper', hi)
    n_inside, n_total, coverage, _ = calculate_coverage(fit)
    add('CrI_points_inside', n_inside)
    add('CrI_total_points', n_total)
    add('CrI_coverage_percent', coverage)
    if diagnostics is not None:
        add('MCMC_mean_chain_acceptance', diagnostics['overall_acceptance'])
        add('MCMC_max_Rhat', diagnostics['max_rhat'])
        for chain, acc in diagnostics['acceptance']:
            add(f'MCMC_chain_{chain}_acceptance', acc)
        if diagnostics['rhat'] is not None:
            for p, value in zip(PARAMETERS, diagnostics['rhat']):
                add(f'Rhat_{p}', value)
        add('MCMC_convergence_assessment', diagnostics['convergence'])
    loo_metrics = calculate_loo_metrics(fit, loo)
    if loo_metrics is not None:
        add('LOO_n', loo_metrics['n'])
        add('LOO_Pearson_r', loo_metrics['pearson_r'])
        add('LOO_RMSE', loo_metrics['rmse'])
    summary = pd.DataFrame(rows)
    summary.to_csv(OUTPUT_DIR / 'RSV_Publication_Summary.csv', index=False)
    with open(OUTPUT_DIR / 'RSV_Publication_Summary.txt', 'w', encoding='utf-8') as f:
        f.write('=' * 100 + '\n')
        f.write('RSV LYON - PUBLICATION ANALYSIS\n')
        f.write('=' * 100 + '\n\n')
        for row in rows:
            f.write(f"{row['metric']:<60}{row['value']}\n")
    if pstats is not None:
        pstats.to_csv(OUTPUT_DIR / 'RSV_Posterior_Parameter_Statistics.csv', index=False)
    return summary

def main():
    print('\n')
    print('=' * 100)
    print('RSV LYON - COMPLETE PUBLICATION ANALYSIS')
    print('=' * 100)
    print('\nThis analysis reads the existing Fortran outputs.')
    fit = load_fit()
    posterior = load_posterior()
    post_perf = load_posterior_performance()
    mle = load_mle()
    diagnostics = load_diagnostics()
    loo = load_loo()
    parameter_summary = load_parameter_summary()
    plot_parameter_posteriors(posterior)
    plot_posterior_performance(post_perf)
    plot_acceptance(diagnostics)
    plot_rhat(diagnostics)
    plot_residuals(fit)
    plot_observed_vs_fitted(fit)
    plot_loo(fit)
    print('\n')
    print('=' * 100)
    print('ANALYSIS COMPLETE')
    print('=' * 100)
    print(f'\nOutput directory:\n  {OUTPUT_DIR.resolve()}')
    print('\nGenerated files:')
    for p in sorted(OUTPUT_DIR.iterdir()):
        print(f'  {p.name}')
if __name__ == '__main__':
    main()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
FIT_FILE = Path('RSV_Fit_Results.csv')

def load_fit():
    """Read the RSV fit CSV and return a dict with required arrays."""
    if not FIT_FILE.exists():
        raise FileNotFoundError(f'Required file not found: {FIT_FILE}')
    df = pd.read_csv(FIT_FILE)
    df.columns = df.columns.str.strip()
    aliases = {'week': ['week'], 'r_data': ['r_data'], 'mle_curve': ['mle_curve'], 'posterior_median': ['posterior_median', 'post_median'], 'posterior_lower': ['posterior_lower', 'post_lower'], 'posterior_upper': ['posterior_upper', 'post_upper']}
    selected = {}
    for standard, possible in aliases.items():
        found = None
        for name in possible:
            if name in df.columns:
                found = name
                break
        if found is None:
            raise ValueError(f"Missing column for '{standard}'. Available: {list(df.columns)}")
        selected[standard] = found
    return {'week': df[selected['week']].to_numpy(float), 'r_data': df[selected['r_data']].to_numpy(float), 'mle_curve': df[selected['mle_curve']].to_numpy(float), 'posterior_median': df[selected['posterior_median']].to_numpy(float), 'posterior_lower': df[selected['posterior_lower']].to_numpy(float), 'posterior_upper': df[selected['posterior_upper']].to_numpy(float)}
fit = load_fit()
weeks_data = fit['week']
r_data = fit['r_data']
mle_curve = fit['mle_curve']
post_median = fit['posterior_median']
lower_ci = fit['posterior_lower']
upper_ci = fit['posterior_upper']
plt.figure(figsize=(11, 7))
plt.fill_between(weeks_data, lower_ci, upper_ci, color='red', alpha=0.25, label='95% Credible Interval')
plt.plot(weeks_data, r_data, 'o:', color='darkgreen', linewidth=2, markersize=7, label='Observed RSV Data')
plt.plot(weeks_data, post_median, '--', color='black', linewidth=2.5, label='Posterior Median')
ax = plt.gca()
ax.axhline(1, linestyle=':', color='black', linewidth=1.0, label='R(t)=1')
xmin = weeks_data.min()
xmax = weeks_data.max()
tick_positions = np.linspace(xmin, xmax, 6)
tick_labels = ['45', '47', '49', '51', '1', '3']
plt.xticks(tick_positions, tick_labels, fontsize=15)
ymin = np.floor(min(lower_ci.min(), r_data.min()) * 5) / 5
ymax = np.ceil(max(upper_ci.max(), r_data.max()) * 5) / 5
plt.yticks(np.arange(ymin, ymax + 0.2, 0.4), fontsize=15)
plt.xlabel('Weeks', fontsize=18)
plt.ylabel('Effective Reproduction Number $R(t)$', fontsize=18)
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
ax.tick_params(axis='x', labelsize=17, width=2.5, length=7)
ax.tick_params(axis='y', labelsize=17, width=2.5, length=7)
plt.grid(alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('RSV_Bayesian_Posterior_Lyon.png', dpi=300)
plt.show()


In [ ]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
FIT_FILE = Path('Influenza_Fit_Results.csv')
MLE_FILE = Path('Influenza_MLE_Summary.dat')
POSTERIOR_FILE = Path('Influenza_Posterior_Samples.dat')
POSTERIOR_PERFORMANCE_FILE = Path('Influenza_Posterior_Performance.dat')
MCMC_DIAGNOSTICS_FILE = Path('Influenza_MCMC_Diagnostics.dat')
OUT_DIR = Path('INFLUENZA_PUBLICATION_ANALYSIS')
OUT_DIR.mkdir(exist_ok=True)
PARAMETERS = ['alpha', 'season_amp', 'sigma', 'shift_weeks']
plt.rcParams.update({'font.family': 'serif', 'font.size': 12, 'axes.linewidth': 1.1, 'figure.dpi': 150, 'savefig.dpi': 300, 'xtick.direction': 'out', 'ytick.direction': 'out'})

def rmse(observed, fitted):
    observed = np.asarray(observed, dtype=float)
    fitted = np.asarray(fitted, dtype=float)
    mask = np.isfinite(observed) & np.isfinite(fitted)
    if np.sum(mask) == 0:
        return np.nan
    return float(np.sqrt(np.mean((observed[mask] - fitted[mask]) ** 2)))

def pearson_r(observed, fitted):
    observed = np.asarray(observed, dtype=float)
    fitted = np.asarray(fitted, dtype=float)
    mask = np.isfinite(observed) & np.isfinite(fitted)
    if np.sum(mask) < 2:
        return np.nan
    x = observed[mask]
    y = fitted[mask]
    if np.std(x) == 0.0:
        return np.nan
    if np.std(y) == 0.0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])

def q95(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return (np.nan, np.nan, np.nan)
    return tuple(np.percentile(x, [2.5, 50.0, 97.5]))

def load_fit():
    if not FIT_FILE.exists():
        raise FileNotFoundError(f'Missing file: {FIT_FILE}')
    df = pd.read_csv(FIT_FILE)
    required = ['week', 'observed', 'mle', 'posterior_median', 'posterior_lower', 'posterior_upper']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError('Influenza_Fit_Results.csv is missing columns: ' + str(missing))
    return df

def load_mle():
    if not MLE_FILE.exists():
        return None
    rows = []
    for line in MLE_FILE.read_text(errors='ignore').splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith('#'):
            continue
        try:
            values = [float(x.replace('D', 'E').replace('d', 'e')) for x in line.split()]
            rows.append(values)
        except ValueError:
            continue
    if len(rows) == 0:
        return None
    x = rows[-1]
    return {'alpha': x[0], 'season_amp': x[1], 'sigma': x[2], 'shift_weeks': x[3], 'logLik': x[4], 'RMSE': x[5], 'Pearson_r': x[6]}

def load_posterior():
    if not POSTERIOR_FILE.exists():
        return None
    data = np.loadtxt(POSTERIOR_FILE, comments='#')
    data = np.atleast_2d(data)
    if data.shape[1] < 4:
        raise ValueError('Posterior sample file must contain four parameter columns.')
    return data[:, :4]

def load_posterior_performance():
    if not POSTERIOR_PERFORMANCE_FILE.exists():
        return None
    data = np.loadtxt(POSTERIOR_PERFORMANCE_FILE, comments='#')
    data = np.atleast_2d(data)
    if data.shape[1] < 2:
        raise ValueError('Posterior performance file must contain Pearson_r and RMSE.')
    return {'pearson_r': data[:, 0], 'rmse': data[:, 1]}

def load_mcmc_diagnostics():
    if not MCMC_DIAGNOSTICS_FILE.exists():
        return None
    lines = MCMC_DIAGNOSTICS_FILE.read_text(errors='ignore').splitlines()
    overall_acceptance = np.nan
    posterior_samples = np.nan
    max_rhat = np.nan
    convergence = None
    walker_ids = []
    walker_acceptance = []
    rhat = None
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if line.startswith('#'):
            continue
        parts = line.split()
        if len(parts) == 4:
            try:
                overall_acceptance = float(parts[0].replace('D', 'E').replace('d', 'e'))
                posterior_samples = float(parts[1].replace('D', 'E').replace('d', 'e'))
                max_rhat = float(parts[2].replace('D', 'E').replace('d', 'e'))
                convergence = parts[3]
                continue
            except ValueError:
                pass
        if len(parts) == 2:
            try:
                walker_id = int(parts[0])
                acceptance = float(parts[1].replace('D', 'E').replace('d', 'e'))
                walker_ids.append(walker_id)
                walker_acceptance.append(acceptance)
                continue
            except ValueError:
                pass
        if len(parts) == 4:
            try:
                vals = np.array([float(x.replace('D', 'E').replace('d', 'e')) for x in parts])
                if np.all(np.isfinite(vals)):
                    rhat = vals
            except ValueError:
                pass
    return {'overall_acceptance': overall_acceptance, 'posterior_samples': posterior_samples, 'max_rhat': max_rhat, 'convergence': convergence, 'walker_ids': np.asarray(walker_ids), 'walker_acceptance': np.asarray(walker_acceptance), 'rhat': rhat}

def parameter_summary(posterior):
    result = {}
    for j, name in enumerate(PARAMETERS):
        lo, med, hi = q95(posterior[:, j])
        result[name] = {'lower': lo, 'median': med, 'upper': hi, 'mean': np.mean(posterior[:, j]), 'sd': np.std(posterior[:, j], ddof=1)}
    return result

def print_results(fit, mle, posterior, performance, diagnostics):
    observed = fit['observed'].to_numpy()
    mle_curve = fit['mle'].to_numpy()
    posterior_median = fit['posterior_median'].to_numpy()
    print()
    print('=' * 90)
    print('INFLUENZA LYON — PERIODIC MODEL')
    print('=' * 90)
    print()
    print('Observations:', len(observed))
    print()
    print('1. MAXIMUM-LIKELIHOOD ESTIMATE')
    print('-' * 90)
    if mle is not None:
        print(f"alpha          = {mle['alpha']:.10f}")
        print(f"season_amp     = {mle['season_amp']:.10f}")
        print(f"sigma          = {mle['sigma']:.10f}")
        print(f"shift_weeks    = {mle['shift_weeks']:.10f}")
        print(f"logLik         = {mle['logLik']:.10f}")
        print(f"RMSE           = {mle['RMSE']:.10f}")
        print(f"Pearson r      = {mle['Pearson_r']:.10f}")
    print()
    print('2. POSTERIOR PARAMETER ESTIMATES')
    print('-' * 90)
    summary = parameter_summary(posterior)
    print(f"{'Parameter':<18}{'Median':>16}{'2.5%':>16}{'97.5%':>16}")
    for name in PARAMETERS:
        s = summary[name]
        print(f"{name:<18}{s['median']:>16.8f}{s['lower']:>16.8f}{s['upper']:>16.8f}")
    print()
    print('3. TRAJECTORY FIT')
    print('-' * 90)
    mle_r = pearson_r(observed, mle_curve)
    mle_rmse = rmse(observed, mle_curve)
    posterior_r = pearson_r(observed, posterior_median)
    posterior_rmse = rmse(observed, posterior_median)
    print(f'MLE Pearson r             = {mle_r:.8f}')
    print(f'MLE RMSE                  = {mle_rmse:.8f}')
    print(f'Posterior median Pearson r = {posterior_r:.8f}')
    print(f'Posterior median RMSE      = {posterior_rmse:.8f}')
    print()
    print('4. POSTERIOR PERFORMANCE DISTRIBUTIONS')
    print('-' * 90)
    if performance is not None:
        rlo, rmed, rhi = q95(performance['pearson_r'])
        elo, emed, ehi = q95(performance['rmse'])
        print(f'Pearson r median = {rmed:.8f}')
        print(f'Pearson r 95% CrI = [{rlo:.8f}, {rhi:.8f}]')
        print(f'RMSE median = {emed:.8f}')
        print(f'RMSE 95% CrI = [{elo:.8f}, {ehi:.8f}]')
    lower = fit['posterior_lower'].to_numpy()
    upper = fit['posterior_upper'].to_numpy()
    inside = (observed >= lower) & (observed <= upper)
    coverage = 100.0 * np.sum(inside) / len(observed)
    print()
    print('5. POSTERIOR TRAJECTORY COVERAGE')
    print('-' * 90)
    print(f'Observed points inside 95% CrI: {np.sum(inside)}/{len(observed)}')
    print(f'Coverage = {coverage:.2f}%')
    print()
    print('6. MCMC DIAGNOSTICS')
    print('-' * 90)
    if diagnostics is not None:
        print(f"Overall acceptance = {diagnostics['overall_acceptance']:.8f}")
        print(f"Posterior samples = {int(diagnostics['posterior_samples'])}")
        print(f"Maximum R-hat = {diagnostics['max_rhat']:.8f}")
        print(f"Convergence flag = {diagnostics['convergence']}")
        if diagnostics['rhat'] is not None:
            print()
            print('Split R-hat:')
            for name, value in zip(PARAMETERS, diagnostics['rhat']):
                print(f'  {name:<15} {value:.8f}')
    print()
    print('7. POSTERIOR PARAMETER CORRELATIONS')
    print('-' * 90)
    corr = np.corrcoef(posterior.T)
    print(f"{'':<18}" + ''.join((f'{x:>16}' for x in PARAMETERS)))
    for i, name in enumerate(PARAMETERS):
        print(f'{name:<18}' + ''.join((f'{corr[i, j]:>16.4f}' for j in range(4))))
    print()
    print('=' * 90)

def plot_main_trajectory(fit):
    week = fit['week'].to_numpy()
    observed = fit['observed'].to_numpy()
    mle = fit['mle'].to_numpy()
    median = fit['posterior_median'].to_numpy()
    lower = fit['posterior_lower'].to_numpy()
    upper = fit['posterior_upper'].to_numpy()
    fig, ax = plt.subplots(figsize=(9.0, 5.8))
    ax.fill_between(week, lower, upper, alpha=0.25, label='95% posterior CrI')
    ax.plot(week, observed, 'o', markersize=5.5, label='Observed $R_{\\mathrm{eff}}$')
    ax.plot(week, mle, '-', linewidth=2.0, label='MLE')
    ax.plot(week, median, '--', linewidth=2.0, label='Posterior median')
    ax.axhline(1.0, linestyle=':', linewidth=1.2)
    ax.text(week[-1], 1.0, '  $R_{\\mathrm{eff}}=1$', va='bottom', ha='right')
    ax.set_xlabel('Week')
    ax.set_ylabel('$R_{\\mathrm{eff}}$')
    ax.set_title('Influenza transmission dynamics under the periodic model')
    ax.legend(frameon=False, loc='best')
    ax.grid(alpha=0.18)
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'Figure_1_Influenza_R_eff_periodic.png', bbox_inches='tight')
    fig.savefig(OUT_DIR / 'Figure_1_Influenza_R_eff_periodic.pdf', bbox_inches='tight')
    plt.close(fig)

def plot_parameter_posteriors(posterior):
    fig, axes = plt.subplots(2, 2, figsize=(9.0, 7.0))
    axes = axes.ravel()
    for j, name in enumerate(PARAMETERS):
        ax = axes[j]
        x = posterior[:, j]
        lo, med, hi = q95(x)
        ax.hist(x, bins=40, density=True, alpha=0.75)
        ax.axvline(med, linestyle='--', linewidth=1.5)
        ax.axvline(lo, linestyle=':', linewidth=1.0)
        ax.axvline(hi, linestyle=':', linewidth=1.0)
        ax.set_xlabel(name)
        ax.set_ylabel('Density')
        ax.set_title(f'Median = {med:.4f}\n95% CrI = [{lo:.4f}, {hi:.4f}]')
        ax.grid(alpha=0.15)
    fig.suptitle('Posterior parameter distributions')
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'Figure_2_posterior_parameters.png', bbox_inches='tight')
    fig.savefig(OUT_DIR / 'Figure_2_posterior_parameters.pdf', bbox_inches='tight')
    plt.close(fig)

def plot_parameter_correlations(posterior):
    corr = np.corrcoef(posterior.T)
    fig, ax = plt.subplots(figsize=(7.0, 6.0))
    im = ax.imshow(corr, vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(np.arange(4))
    ax.set_yticks(np.arange(4))
    ax.set_xticklabels(PARAMETERS, rotation=30, ha='right')
    ax.set_yticklabels(PARAMETERS)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{corr[i, j]:.2f}', ha='center', va='center')
    ax.set_title('Posterior parameter correlations')
    fig.colorbar(im, ax=ax, label='Pearson correlation')
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'Figure_3_parameter_correlation.png', bbox_inches='tight')
    fig.savefig(OUT_DIR / 'Figure_3_parameter_correlation.pdf', bbox_inches='tight')
    plt.close(fig)

def plot_performance(performance):
    if performance is None:
        return
    fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.5))
    r = performance['pearson_r']
    rlo, rmed, rhi = q95(r)
    axes[0].hist(r, bins=40, density=True, alpha=0.75)
    axes[0].axvline(rmed, linestyle='--', linewidth=1.5)
    axes[0].set_xlabel('Pearson correlation, $r$')
    axes[0].set_ylabel('Density')
    axes[0].set_title(f'Median = {rmed:.4f}\n95% CrI = [{rlo:.4f}, {rhi:.4f}]')
    e = performance['rmse']
    elo, emed, ehi = q95(e)
    axes[1].hist(e, bins=40, density=True, alpha=0.75)
    axes[1].axvline(emed, linestyle='--', linewidth=1.5)
    axes[1].set_xlabel('RMSE')
    axes[1].set_ylabel('Density')
    axes[1].set_title(f'Median = {emed:.4f}\n95% CrI = [{elo:.4f}, {ehi:.4f}]')
    fig.suptitle('Posterior trajectory-fit diagnostics')
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'Figure_4_posterior_fit_diagnostics.png', bbox_inches='tight')
    fig.savefig(OUT_DIR / 'Figure_4_posterior_fit_diagnostics.pdf', bbox_inches='tight')
    plt.close(fig)

def plot_acceptance(diagnostics):
    if diagnostics is None:
        return
    if len(diagnostics['walker_ids']) == 0:
        return
    walkers = diagnostics['walker_ids']
    acceptance = diagnostics['walker_acceptance']
    fig, ax = plt.subplots(figsize=(9.0, 4.8))
    ax.bar(walkers, acceptance)
    overall = diagnostics['overall_acceptance']
    if np.isfinite(overall):
        ax.axhline(overall, linestyle='--', linewidth=1.5, label=f'Overall = {overall:.3f}')
    ax.set_xlabel('Walker')
    ax.set_ylabel('Acceptance fraction')
    ax.set_title('MCMC walker acceptance fractions')
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'Figure_5_MCMC_acceptance.png', bbox_inches='tight')
    fig.savefig(OUT_DIR / 'Figure_5_MCMC_acceptance.pdf', bbox_inches='tight')
    plt.close(fig)

def plot_rhat(diagnostics):
    if diagnostics is None:
        return
    rhat = diagnostics['rhat']
    if rhat is None:
        return
    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    x = np.arange(len(PARAMETERS))
    ax.bar(x, rhat)
    ax.axhline(1.01, linestyle='--', linewidth=1.5, label='$\\hat{R}=1.01$')
    ax.axhline(1.05, linestyle=':', linewidth=1.5, label='$\\hat{R}=1.05$')
    ax.set_xticks(x)
    ax.set_xticklabels(PARAMETERS)
    ax.set_ylabel('Split $\\hat{R}$')
    ax.set_title('MCMC convergence diagnostic')
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'Figure_6_Rhat.png', bbox_inches='tight')
    fig.savefig(OUT_DIR / 'Figure_6_Rhat.pdf', bbox_inches='tight')
    plt.close(fig)

def plot_residuals(fit):
    week = fit['week'].to_numpy()
    observed = fit['observed'].to_numpy()
    mle = fit['mle'].to_numpy()
    posterior = fit['posterior_median'].to_numpy()
    residual_mle = observed - mle
    residual_post = observed - posterior
    fig, ax = plt.subplots(figsize=(9.0, 5.2))
    ax.axhline(0.0, linestyle='--', linewidth=1.0)
    ax.plot(week, residual_mle, 'o-', linewidth=1.5, label='MLE')
    ax.plot(week, residual_post, 's--', linewidth=1.5, label='Posterior median')
    ax.set_xlabel('Week')
    ax.set_ylabel('Observed - fitted')
    ax.set_title('Influenza model residuals')
    ax.legend(frameon=False)
    ax.grid(alpha=0.15)
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'Figure_7_residuals.png', bbox_inches='tight')
    fig.savefig(OUT_DIR / 'Figure_7_residuals.pdf', bbox_inches='tight')
    plt.close(fig)

def plot_observed_vs_fitted(fit):
    observed = fit['observed'].to_numpy()
    mle = fit['mle'].to_numpy()
    posterior = fit['posterior_median'].to_numpy()
    values = np.concatenate([observed, mle, posterior])
    vmin = np.nanmin(values)
    vmax = np.nanmax(values)
    pad = 0.05 * (vmax - vmin)
    fig, ax = plt.subplots(figsize=(6.5, 6.0))
    ax.plot([vmin - pad, vmax + pad], [vmin - pad, vmax + pad], '--', linewidth=1.2, label='1:1')
    ax.scatter(observed, mle, s=38, label='MLE')
    ax.scatter(observed, posterior, s=38, marker='s', label='Posterior median')
    ax.set_xlabel('Observed $R_{\\mathrm{eff}}$')
    ax.set_ylabel('Fitted $R_{\\mathrm{eff}}$')
    ax.set_title('Observed versus fitted $R_{\\mathrm{eff}}$')
    ax.legend(frameon=False)
    ax.grid(alpha=0.15)
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'Figure_8_observed_vs_fitted.png', bbox_inches='tight')
    fig.savefig(OUT_DIR / 'Figure_8_observed_vs_fitted.pdf', bbox_inches='tight')
    plt.close(fig)

def save_summary(fit, mle, posterior, performance, diagnostics):
    observed = fit['observed'].to_numpy()
    mle_curve = fit['mle'].to_numpy()
    posterior_curve = fit['posterior_median'].to_numpy()
    rows = []

    def add(name, value):
        rows.append({'metric': name, 'value': value})
    add('number_of_observations', len(observed))
    add('MLE_Pearson_r', pearson_r(observed, mle_curve))
    add('MLE_RMSE', rmse(observed, mle_curve))
    add('Posterior_median_Pearson_r', pearson_r(observed, posterior_curve))
    add('Posterior_median_RMSE', rmse(observed, posterior_curve))
    ps = parameter_summary(posterior)
    for name in PARAMETERS:
        add(f'{name}_posterior_median', ps[name]['median'])
        add(f'{name}_posterior_2.5_percent', ps[name]['lower'])
        add(f'{name}_posterior_97.5_percent', ps[name]['upper'])
    if performance is not None:
        rlo, rmed, rhi = q95(performance['pearson_r'])
        elo, emed, ehi = q95(performance['rmse'])
        add('Posterior_Pearson_r_median', rmed)
        add('Posterior_Pearson_r_2.5_percent', rlo)
        add('Posterior_Pearson_r_97.5_percent', rhi)
        add('Posterior_RMSE_median', emed)
        add('Posterior_RMSE_2.5_percent', elo)
        add('Posterior_RMSE_97.5_percent', ehi)
    lower = fit['posterior_lower'].to_numpy()
    upper = fit['posterior_upper'].to_numpy()
    inside = (observed >= lower) & (observed <= upper)
    add('latent_95_CrI_coverage_percent', 100.0 * np.sum(inside) / len(observed))
    if diagnostics is not None:
        add('MCMC_overall_acceptance', diagnostics['overall_acceptance'])
        add('MCMC_posterior_samples', diagnostics['posterior_samples'])
        add('MCMC_max_Rhat', diagnostics['max_rhat'])
        if diagnostics['rhat'] is not None:
            for name, value in zip(PARAMETERS, diagnostics['rhat']):
                add(f'Rhat_{name}', value)
    summary = pd.DataFrame(rows)
    summary.to_csv(OUT_DIR / 'Influenza_Publication_Summary.csv', index=False)
    with open(OUT_DIR / 'Influenza_Publication_Summary.txt', 'w') as f:
        f.write('INFLUENZA LYON — PERIODIC MODEL\n')
        f.write('=' * 80 + '\n\n')
        for row in rows:
            f.write(f"{row['metric']:<50}{row['value']}\n")

def main():
    print()
    print('Loading current periodic-model results...')
    fit = load_fit()
    mle = load_mle()
    posterior = load_posterior()
    performance = load_posterior_performance()
    diagnostics = load_mcmc_diagnostics()
    if len(posterior) == 0:
        raise RuntimeError('No posterior samples found.')
    if len(fit) != 13:
        print(f'WARNING: Expected 13 observations, found {len(fit)}.')
    print_results(fit, mle, posterior, performance, diagnostics)
    print()
    print('Generating publication figures...')
    plot_main_trajectory(fit)
    plot_parameter_posteriors(posterior)
    plot_parameter_correlations(posterior)
    plot_performance(performance)
    plot_acceptance(diagnostics)
    plot_rhat(diagnostics)
    plot_residuals(fit)
    plot_observed_vs_fitted(fit)
    save_summary(fit, mle, posterior, performance, diagnostics)
    print()
    print('=' * 90)
    print('ANALYSIS COMPLETED')
    print('=' * 90)
    print()
    print(f'Output directory: {OUT_DIR.resolve()}')
    print()
    for file in sorted(OUT_DIR.iterdir()):
        print(f'  {file.name}')
    print()
if __name__ == '__main__':
    main()


In [ ]:
from __future__ import annotations
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
FIT_FILE = Path('Influenza_Fit_Results.csv')
MLE_FILE = Path('Influenza_MLE_Summary.dat')
POSTERIOR_FILE = Path('Influenza_Posterior_Samples.dat')
POSTERIOR_PERFORMANCE_FILE = Path('Influenza_Posterior_Performance.dat')
MCMC_DIAGNOSTICS_FILE = Path('Influenza_MCMC_Diagnostics.dat')
OUT_DIR = Path('INFLUENZA_PUBLICATION_ANALYSIS')
OUT_DIR.mkdir(parents=True, exist_ok=True)
PARAMETERS = ['alpha', 'season_amp', 'sigma', 'shift_weeks']
FIT_COLUMNS = ['week', 'observed', 'mle', 'posterior_median', 'posterior_lower', 'posterior_upper']
plt.rcParams.update({'font.family': 'serif', 'font.size': 12, 'axes.linewidth': 1.1, 'figure.dpi': 150, 'savefig.dpi': 300, 'xtick.direction': 'out', 'ytick.direction': 'out'})

def finite_pair(observed, fitted):
    """Return paired finite values from two equal-length arrays."""
    observed = np.asarray(observed, dtype=float).ravel()
    fitted = np.asarray(fitted, dtype=float).ravel()
    if observed.size != fitted.size:
        raise ValueError(f'Array-length mismatch: observed={observed.size}, fitted={fitted.size}.')
    mask = np.isfinite(observed) & np.isfinite(fitted)
    return (observed[mask], fitted[mask])

def rmse(observed, fitted):
    """Root mean squared error using paired finite values."""
    observed, fitted = finite_pair(observed, fitted)
    if observed.size == 0:
        return np.nan
    return float(np.sqrt(np.mean((observed - fitted) ** 2)))

def pearson_r(observed, fitted):
    """Pearson correlation using paired finite values."""
    observed, fitted = finite_pair(observed, fitted)
    if observed.size < 2:
        return np.nan
    if np.std(observed) == 0.0 or np.std(fitted) == 0.0:
        return np.nan
    return float(np.corrcoef(observed, fitted)[0, 1])

def q95(values):
    """Return the 2.5th percentile, median and 97.5th percentile."""
    values = np.asarray(values, dtype=float).ravel()
    values = values[np.isfinite(values)]
    if values.size == 0:
        return (np.nan, np.nan, np.nan)
    lo, med, hi = np.percentile(values, [2.5, 50.0, 97.5])
    return (float(lo), float(med), float(hi))

def parse_float(token):
    """Convert ordinary or Fortran scientific notation to float."""
    return float(token.strip().replace('D', 'E').replace('d', 'e'))

def load_fit():
    if not FIT_FILE.exists():
        raise FileNotFoundError(f'Missing fit file: {FIT_FILE.resolve()}')
    number_pattern = re.compile('[-+]?(?:\\d+\\.?\\d*|\\.\\d+)(?:[EeDd][-+]?\\d+)?')
    rows = []
    with FIT_FILE.open('r', encoding='utf-8-sig') as file:
        for line_number, raw_line in enumerate(file, start=1):
            line = raw_line.strip()
            if not line or line.startswith('#'):
                continue
            if not number_pattern.search(line):
                continue
            tokens = number_pattern.findall(line)
            if len(tokens) != len(FIT_COLUMNS):
                if any((char.isalpha() for char in line)):
                    continue
                raise ValueError(f'Malformed fit-data row on line {line_number}.\nExpected {len(FIT_COLUMNS)} values, found {len(tokens)}.\nContent: {raw_line.rstrip()}')
            rows.append([parse_float(x) for x in tokens])
    if not rows:
        raise ValueError(f'No numerical six-column data rows found in {FIT_FILE}.')
    df = pd.DataFrame(rows, columns=FIT_COLUMNS)
    finite_mask = np.isfinite(df[FIT_COLUMNS].to_numpy(dtype=float)).all(axis=1)
    if not finite_mask.all():
        print(f'WARNING: Excluding {np.sum(~finite_mask)} rows with non-finite values.')
        df = df.loc[finite_mask].copy()
    df = df.sort_values('week').reset_index(drop=True)
    if df.empty:
        raise ValueError('No valid fit-data rows remain.')
    if (df['posterior_lower'] > df['posterior_upper']).any():
        raise ValueError('Some posterior lower limits exceed upper limits. Check the ordering of the last two columns.')
    print('\nFIT-FILE DIAGNOSTICS')
    print('-' * 75)
    print('File:', FIT_FILE.resolve())
    print('Columns:', list(df.columns))
    print('Valid observations:', len(df))
    print('Week range:', df['week'].min(), 'to', df['week'].max())
    print('\nFirst five rows:')
    print(df.head().to_string(index=False))
    return df

def load_mle():
    if not MLE_FILE.exists():
        print(f'Optional MLE summary not found: {MLE_FILE}')
        return None
    rows = []
    for line in MLE_FILE.read_text(encoding='utf-8', errors='ignore').splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        try:
            values = [parse_float(x) for x in line.split()]
            rows.append(values)
        except ValueError:
            continue
    if not rows:
        return None
    x = rows[-1]
    if len(x) < 7:
        warnings.warn('MLE summary has fewer than seven values. Expected alpha, season_amp, sigma, shift_weeks, logLik, RMSE, Pearson_r.')
        return None
    return {'alpha': x[0], 'season_amp': x[1], 'sigma': x[2], 'shift_weeks': x[3], 'logLik': x[4], 'RMSE': x[5], 'Pearson_r': x[6]}

def load_posterior():
    if not POSTERIOR_FILE.exists():
        raise FileNotFoundError(f'Required posterior file not found: {POSTERIOR_FILE.resolve()}')
    data = np.loadtxt(POSTERIOR_FILE, comments='#', ndmin=2)
    if data.shape[1] < 4:
        raise ValueError('Posterior sample file must contain at least four columns.')
    posterior = np.asarray(data[:, :4], dtype=float)
    valid_rows = np.isfinite(posterior).all(axis=1)
    if not valid_rows.all():
        print(f'WARNING: Excluding {np.sum(~valid_rows)} posterior samples with non-finite parameter values.')
        posterior = posterior[valid_rows]
    if posterior.shape[0] == 0:
        raise ValueError('No valid posterior samples remain.')
    print('\nPOSTERIOR SAMPLE DIAGNOSTICS')
    print('-' * 75)
    print('Valid posterior samples:', len(posterior))
    print('Number of parameters:', posterior.shape[1])
    return posterior

def load_posterior_performance():
    if not POSTERIOR_PERFORMANCE_FILE.exists():
        print('Optional posterior performance file not found:', POSTERIOR_PERFORMANCE_FILE)
        return None
    data = np.loadtxt(POSTERIOR_PERFORMANCE_FILE, comments='#', ndmin=2)
    if data.shape[1] < 2:
        raise ValueError('Posterior performance file must contain Pearson_r and RMSE columns.')
    r = np.asarray(data[:, 0], dtype=float)
    errors = np.asarray(data[:, 1], dtype=float)
    valid = np.isfinite(r) & np.isfinite(errors)
    if not valid.all():
        print(f'WARNING: Excluding {np.sum(~valid)} invalid posterior performance rows.')
    r = r[valid]
    errors = errors[valid]
    if r.size == 0:
        warnings.warn('No valid posterior performance samples remain.')
        return None
    return {'pearson_r': r, 'rmse': errors}

def load_mcmc_diagnostics():
    if not MCMC_DIAGNOSTICS_FILE.exists():
        print('Optional MCMC diagnostics file not found:', MCMC_DIAGNOSTICS_FILE)
        return None
    lines = MCMC_DIAGNOSTICS_FILE.read_text(encoding='utf-8', errors='ignore').splitlines()
    overall_acceptance = np.nan
    posterior_samples = np.nan
    max_rhat = np.nan
    convergence = None
    walker_ids = []
    walker_acceptance = []
    rhat = None
    for line in lines:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        parts = line.split()
        if len(parts) == 4:
            try:
                candidate = [parse_float(parts[0]), parse_float(parts[1]), parse_float(parts[2])]
                if np.all(np.isfinite(candidate)):
                    overall_acceptance = candidate[0]
                    posterior_samples = candidate[1]
                    max_rhat = candidate[2]
                    convergence = parts[3]
                    continue
            except ValueError:
                pass
            try:
                values = np.array([parse_float(x) for x in parts], dtype=float)
                if np.all(np.isfinite(values)):
                    rhat = values
            except ValueError:
                pass
            continue
        if len(parts) == 2:
            try:
                walker_id = int(parts[0])
                acceptance = parse_float(parts[1])
                walker_ids.append(walker_id)
                walker_acceptance.append(acceptance)
            except ValueError:
                pass
    return {'overall_acceptance': overall_acceptance, 'posterior_samples': posterior_samples, 'max_rhat': max_rhat, 'convergence': convergence, 'walker_ids': np.asarray(walker_ids), 'walker_acceptance': np.asarray(walker_acceptance), 'rhat': rhat}

def parameter_summary(posterior):
    result = {}
    for j, name in enumerate(PARAMETERS):
        values = posterior[:, j]
        lo, med, hi = q95(values)
        result[name] = {'lower': lo, 'median': med, 'upper': hi, 'mean': float(np.mean(values)), 'sd': float(np.std(values, ddof=1)) if len(values) > 1 else np.nan}
    return result

def calculate_trajectory_metrics(fit):
    observed = fit['observed'].to_numpy(dtype=float)
    mle = fit['mle'].to_numpy(dtype=float)
    posterior_median = fit['posterior_median'].to_numpy(dtype=float)
    observed_mle, fitted_mle = finite_pair(observed, mle)
    observed_post, fitted_post = finite_pair(observed, posterior_median)
    metrics = {'mle_r': pearson_r(observed_mle, fitted_mle), 'mle_rmse': rmse(observed_mle, fitted_mle), 'posterior_r': pearson_r(observed_post, fitted_post), 'posterior_rmse': rmse(observed_post, fitted_post), 'n_mle_pairs': len(observed_mle), 'n_posterior_pairs': len(observed_post)}
    print('\nTRAJECTORY DATA DIAGNOSTICS')
    print('-' * 75)
    for name, values in [('Observed', observed), ('MLE trajectory', mle), ('Posterior median', posterior_median)]:
        finite = np.isfinite(values)
        valid_values = values[finite]
        print(f'\n{name}:')
        print(f'  Total values       = {len(values)}')
        print(f'  Finite values      = {finite.sum()}')
        print(f'  Non-finite values  = {(~finite).sum()}')
        if valid_values.size:
            print(f'  Minimum            = {valid_values.min():.10g}')
            print(f'  Maximum            = {valid_values.max():.10g}')
            print(f'  Standard deviation = {valid_values.std():.10g}')
        else:
            print('  No finite values.')
    print('\nValid observation-fit pairs:')
    print('  MLE:', metrics['n_mle_pairs'])
    print('  Posterior median:', metrics['n_posterior_pairs'])
    if not np.isfinite(metrics['posterior_r']):
        print('\nWARNING: Posterior median Pearson r is undefined. This occurs if fewer than two finite pairs remain or either the observed or posterior-median series is constant.')
    if not np.isfinite(metrics['posterior_rmse']):
        print('\nWARNING: Posterior median RMSE is undefined because there are no finite observation-fit pairs.')
    return metrics

def print_results(fit, mle, posterior, performance, diagnostics, trajectory_metrics):
    observed = fit['observed'].to_numpy(dtype=float)
    print('\n' + '=' * 90)
    print('INFLUENZA LYON — PERIODIC MODEL')
    print('=' * 90)
    print('Number of observations:', len(observed))
    print('\n1. MAXIMUM-LIKELIHOOD ESTIMATE')
    print('-' * 90)
    if mle is not None:
        for key in ['alpha', 'season_amp', 'sigma', 'shift_weeks', 'logLik', 'RMSE', 'Pearson_r']:
            print(f'{key:<18} = {mle[key]:.10f}')
    else:
        print('MLE summary file unavailable or invalid.')
    print('\n2. POSTERIOR PARAMETER ESTIMATES')
    print('-' * 90)
    summary = parameter_summary(posterior)
    print(f"{'Parameter':<18}{'Median':>16}{'2.5%':>16}{'97.5%':>16}")
    for name in PARAMETERS:
        s = summary[name]
        print(f"{name:<18}{s['median']:>16.8f}{s['lower']:>16.8f}{s['upper']:>16.8f}")
    print('\n3. TRAJECTORY FIT')
    print('-' * 90)
    print(f"MLE Pearson r               = {trajectory_metrics['mle_r']:.8f}")
    print(f"MLE RMSE                    = {trajectory_metrics['mle_rmse']:.8f}")
    print(f"Posterior median Pearson r  = {trajectory_metrics['posterior_r']:.8f}")
    print(f"Posterior median RMSE       = {trajectory_metrics['posterior_rmse']:.8f}")
    print('\n4. POSTERIOR PERFORMANCE DISTRIBUTIONS')
    print('-' * 90)
    if performance is not None:
        rlo, rmed, rhi = q95(performance['pearson_r'])
        elo, emed, ehi = q95(performance['rmse'])
        print(f'Pearson r median = {rmed:.8f}')
        print(f'Pearson r 95% CrI = [{rlo:.8f}, {rhi:.8f}]')
        print(f'RMSE median = {emed:.8f}')
        print(f'RMSE 95% CrI = [{elo:.8f}, {ehi:.8f}]')
    else:
        print('Posterior performance file unavailable or empty.')
    lower = fit['posterior_lower'].to_numpy(dtype=float)
    upper = fit['posterior_upper'].to_numpy(dtype=float)
    valid = np.isfinite(observed) & np.isfinite(lower) & np.isfinite(upper)
    inside = (observed[valid] >= lower[valid]) & (observed[valid] <= upper[valid])
    coverage = 100.0 * inside.sum() / valid.sum() if valid.sum() > 0 else np.nan
    print('\n5. POSTERIOR TRAJECTORY COVERAGE')
    print('-' * 90)
    print(f'Valid observed points = {valid.sum()}/{len(observed)}')
    print(f'Points inside 95% CrI = {inside.sum()}/{valid.sum()}')
    print(f'Coverage = {coverage:.2f}%')
    print('\n6. MCMC DIAGNOSTICS')
    print('-' * 90)
    if diagnostics is not None:
        print(f"Overall acceptance = {diagnostics['overall_acceptance']:.8f}")
        samples = diagnostics['posterior_samples']
        if np.isfinite(samples):
            print(f'Posterior samples = {int(samples)}')
        else:
            print('Posterior samples = unavailable')
        print(f"Maximum R-hat = {diagnostics['max_rhat']:.8f}")
        print(f"Convergence flag = {diagnostics['convergence']}")
        if diagnostics['rhat'] is not None:
            print('\nSplit R-hat:')
            for name, value in zip(PARAMETERS, diagnostics['rhat']):
                print(f'  {name:<15} {value:.8f}')
    else:
        print('MCMC diagnostics unavailable.')
    print('\n7. POSTERIOR PARAMETER CORRELATIONS')
    print('-' * 90)
    corr = np.corrcoef(posterior.T)
    print(f"{'':<18}" + ''.join((f'{x:>16}' for x in PARAMETERS)))
    for i, name in enumerate(PARAMETERS):
        print(f'{name:<18}' + ''.join((f'{corr[i, j]:>16.4f}' for j in range(4))))
    print('\n' + '=' * 90)

def plot_main_trajectory(fit):
    week = fit['week'].to_numpy(dtype=float)
    observed = fit['observed'].to_numpy(dtype=float)
    mle = fit['mle'].to_numpy(dtype=float)
    median = fit['posterior_median'].to_numpy(dtype=float)
    lower = fit['posterior_lower'].to_numpy(dtype=float)
    upper = fit['posterior_upper'].to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(9.0, 5.8))
    ax.fill_between(week, lower, upper, alpha=0.25, label='95% posterior CrI')
    ax.plot(week, observed, 'o', markersize=5.5, label='Observed $R_{\\mathrm{eff}}$')
    ax.plot(week, mle, '-', linewidth=2.0, label='MLE')
    ax.plot(week, median, '--', linewidth=2.0, label='Posterior median')
    ax.axhline(1.0, linestyle=':', linewidth=1.2, color='black')
    ax.text(week[-1], 1.0, '  $R_{\\mathrm{eff}}=1$', va='bottom', ha='right')
    ax.set_xlabel('Week')
    ax.set_ylabel('$R_{\\mathrm{eff}}$')
    ax.set_title('Influenza transmission dynamics under the periodic model')
    ax.legend(frameon=False, loc='best')
    ax.grid(alpha=0.18)
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'Figure_1_Influenza_R_eff_periodic.png', bbox_inches='tight')
    fig.savefig(OUT_DIR / 'Figure_1_Influenza_R_eff_periodic.pdf', bbox_inches='tight')
    plt.close(fig)

def plot_parameter_posteriors(posterior):
    fig, axes = plt.subplots(2, 2, figsize=(9.0, 7.0))
    axes = axes.ravel()
    for j, name in enumerate(PARAMETERS):
        ax = axes[j]
        values = posterior[:, j]
        lo, med, hi = q95(values)
        ax.hist(values, bins=40, density=True, alpha=0.75)
        ax.axvline(med, linestyle='--', linewidth=1.5)
        ax.axvline(lo, linestyle=':', linewidth=1.0)
        ax.axvline(hi, linestyle=':', linewidth=1.0)
        ax.set_xlabel(name)
        ax.set_ylabel('Density')
        ax.set_title(f'Median = {med:.4f}\n95% CrI = [{lo:.4f}, {hi:.4f}]')
        ax.grid(alpha=0.15)
    fig.suptitle('Posterior parameter distributions')
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'Figure_2_posterior_parameters.png', bbox_inches='tight')
    fig.savefig(OUT_DIR / 'Figure_2_posterior_parameters.pdf', bbox_inches='tight')
    plt.close(fig)

def plot_parameter_correlations(posterior):
    corr = np.corrcoef(posterior.T)
    fig, ax = plt.subplots(figsize=(7.0, 6.0))
    im = ax.imshow(corr, vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(np.arange(4))
    ax.set_yticks(np.arange(4))
    ax.set_xticklabels(PARAMETERS, rotation=30, ha='right')
    ax.set_yticklabels(PARAMETERS)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{corr[i, j]:.2f}', ha='center', va='center')
    ax.set_title('Posterior parameter correlations')
    fig.colorbar(im, ax=ax, label='Pearson correlation')
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'Figure_3_parameter_correlation.png', bbox_inches='tight')
    fig.savefig(OUT_DIR / 'Figure_3_parameter_correlation.pdf', bbox_inches='tight')
    plt.close(fig)

def plot_performance(performance):
    if performance is None:
        return
    fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.5))
    r = performance['pearson_r']
    rlo, rmed, rhi = q95(r)
    axes[0].hist(r, bins=40, density=True, alpha=0.75)
    axes[0].axvline(rmed, linestyle='--', linewidth=1.5)
    axes[0].set_xlabel('Pearson correlation, $r$')
    axes[0].set_ylabel('Density')
    axes[0].set_title(f'Median = {rmed:.4f}\n95% CrI = [{rlo:.4f}, {rhi:.4f}]')
    errors = performance['rmse']
    elo, emed, ehi = q95(errors)
    axes[1].hist(errors, bins=40, density=True, alpha=0.75)
    axes[1].axvline(emed, linestyle='--', linewidth=1.5)
    axes[1].set_xlabel('RMSE')
    axes[1].set_ylabel('Density')
    axes[1].set_title(f'Median = {emed:.4f}\n95% CrI = [{elo:.4f}, {ehi:.4f}]')
    fig.suptitle('Posterior trajectory-fit diagnostics')
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'Figure_4_posterior_fit_diagnostics.png', bbox_inches='tight')
    fig.savefig(OUT_DIR / 'Figure_4_posterior_fit_diagnostics.pdf', bbox_inches='tight')
    plt.close(fig)

def plot_acceptance(diagnostics):
    if diagnostics is None:
        return
    walkers = diagnostics['walker_ids']
    acceptance = diagnostics['walker_acceptance']
    if len(walkers) == 0:
        return
    fig, ax = plt.subplots(figsize=(9.0, 4.8))
    ax.bar(walkers, acceptance)
    overall = diagnostics['overall_acceptance']
    if np.isfinite(overall):
        ax.axhline(overall, linestyle='--', linewidth=1.5, label=f'Overall = {overall:.3f}')
    ax.set_xlabel('Walker')
    ax.set_ylabel('Acceptance fraction')
    ax.set_title('MCMC walker acceptance fractions')
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'Figure_5_MCMC_acceptance.png', bbox_inches='tight')
    fig.savefig(OUT_DIR / 'Figure_5_MCMC_acceptance.pdf', bbox_inches='tight')
    plt.close(fig)

def plot_rhat(diagnostics):
    if diagnostics is None:
        return
    rhat = diagnostics['rhat']
    if rhat is None or len(rhat) != len(PARAMETERS):
        return
    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    x = np.arange(len(PARAMETERS))
    ax.bar(x, rhat)
    ax.axhline(1.01, linestyle='--', linewidth=1.5, label='$\\hat{R}=1.01$')
    ax.axhline(1.05, linestyle=':', linewidth=1.5, label='$\\hat{R}=1.05$')
    ax.set_xticks(x)
    ax.set_xticklabels(PARAMETERS)
    ax.set_ylabel('Split $\\hat{R}$')
    ax.set_title('MCMC convergence diagnostic')
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'Figure_6_Rhat.png', bbox_inches='tight')
    fig.savefig(OUT_DIR / 'Figure_6_Rhat.pdf', bbox_inches='tight')
    plt.close(fig)

def plot_residuals(fit):
    week = fit['week'].to_numpy(dtype=float)
    observed = fit['observed'].to_numpy(dtype=float)
    mle = fit['mle'].to_numpy(dtype=float)
    posterior = fit['posterior_median'].to_numpy(dtype=float)
    residual_mle = observed - mle
    residual_post = observed - posterior
    fig, ax = plt.subplots(figsize=(9.0, 5.2))
    ax.axhline(0.0, linestyle='--', linewidth=1.0, color='black')
    ax.plot(week, residual_mle, 'o-', linewidth=1.5, label='MLE')
    ax.plot(week, residual_post, 's--', linewidth=1.5, label='Posterior median')
    ax.set_xlabel('Week')
    ax.set_ylabel('Observed - fitted')
    ax.set_title('Influenza model residuals')
    ax.legend(frameon=False)
    ax.grid(alpha=0.15)
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'Figure_7_residuals.png', bbox_inches='tight')
    fig.savefig(OUT_DIR / 'Figure_7_residuals.pdf', bbox_inches='tight')
    plt.close(fig)

def plot_observed_vs_fitted(fit):
    observed = fit['observed'].to_numpy(dtype=float)
    mle = fit['mle'].to_numpy(dtype=float)
    posterior = fit['posterior_median'].to_numpy(dtype=float)
    valid_mle = np.isfinite(observed) & np.isfinite(mle)
    valid_post = np.isfinite(observed) & np.isfinite(posterior)
    if not valid_mle.any() and (not valid_post.any()):
        warnings.warn('Cannot create observed-versus-fitted figure: no finite pairs are available.')
        return
    arrays = [observed[valid_mle], mle[valid_mle], observed[valid_post], posterior[valid_post]]
    finite_arrays = [x for x in arrays if x.size > 0]
    values = np.concatenate(finite_arrays)
    vmin = np.min(values)
    vmax = np.max(values)
    pad = 0.05 * (vmax - vmin) if vmax > vmin else 0.1
    fig, ax = plt.subplots(figsize=(6.5, 6.0))
    ax.plot([vmin - pad, vmax + pad], [vmin - pad, vmax + pad], '--', linewidth=1.2, label='1:1')
    ax.scatter(observed[valid_mle], mle[valid_mle], s=38, label='MLE')
    ax.scatter(observed[valid_post], posterior[valid_post], s=38, marker='s', label='Posterior median')
    ax.set_xlabel('Observed $R_{\\mathrm{eff}}$')
    ax.set_ylabel('Fitted $R_{\\mathrm{eff}}$')
    ax.set_title('Observed versus fitted $R_{\\mathrm{eff}}$')
    ax.legend(frameon=False)
    ax.grid(alpha=0.15)
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'Figure_8_observed_vs_fitted.png', bbox_inches='tight')
    fig.savefig(OUT_DIR / 'Figure_8_observed_vs_fitted.pdf', bbox_inches='tight')
    plt.close(fig)

def save_summary(fit, mle, posterior, performance, diagnostics, trajectory_metrics):
    rows = []

    def add(name, value):
        rows.append({'metric': name, 'value': value})
    add('number_of_observations', len(fit))
    add('MLE_Pearson_r', trajectory_metrics['mle_r'])
    add('MLE_RMSE', trajectory_metrics['mle_rmse'])
    add('Posterior_median_Pearson_r', trajectory_metrics['posterior_r'])
    add('Posterior_median_RMSE', trajectory_metrics['posterior_rmse'])
    add('Number_of_valid_MLE_pairs', trajectory_metrics['n_mle_pairs'])
    add('Number_of_valid_posterior_pairs', trajectory_metrics['n_posterior_pairs'])
    if mle is not None:
        for name, value in mle.items():
            add(f'MLE_{name}', value)
    ps = parameter_summary(posterior)
    for name in PARAMETERS:
        add(f'{name}_posterior_median', ps[name]['median'])
        add(f'{name}_posterior_2.5_percent', ps[name]['lower'])
        add(f'{name}_posterior_97.5_percent', ps[name]['upper'])
        add(f'{name}_posterior_mean', ps[name]['mean'])
        add(f'{name}_posterior_sd', ps[name]['sd'])
    if performance is not None:
        rlo, rmed, rhi = q95(performance['pearson_r'])
        elo, emed, ehi = q95(performance['rmse'])
        add('Posterior_Pearson_r_median', rmed)
        add('Posterior_Pearson_r_2.5_percent', rlo)
        add('Posterior_Pearson_r_97.5_percent', rhi)
        add('Posterior_RMSE_median', emed)
        add('Posterior_RMSE_2.5_percent', elo)
        add('Posterior_RMSE_97.5_percent', ehi)
    observed = fit['observed'].to_numpy(dtype=float)
    lower = fit['posterior_lower'].to_numpy(dtype=float)
    upper = fit['posterior_upper'].to_numpy(dtype=float)
    valid = np.isfinite(observed) & np.isfinite(lower) & np.isfinite(upper)
    inside = (observed[valid] >= lower[valid]) & (observed[valid] <= upper[valid])
    coverage = 100.0 * inside.sum() / valid.sum() if valid.sum() > 0 else np.nan
    add('latent_95_CrI_coverage_percent', coverage)
    if diagnostics is not None:
        add('MCMC_overall_acceptance', diagnostics['overall_acceptance'])
        add('MCMC_posterior_samples', diagnostics['posterior_samples'])
        add('MCMC_max_Rhat', diagnostics['max_rhat'])
        if diagnostics['rhat'] is not None:
            for name, value in zip(PARAMETERS, diagnostics['rhat']):
                add(f'Rhat_{name}', value)
    summary_df = pd.DataFrame(rows)
    summary_df.to_csv(OUT_DIR / 'Influenza_Publication_Summary.csv', index=False)
    with open(OUT_DIR / 'Influenza_Publication_Summary.txt', 'w', encoding='utf-8') as f:
        f.write('INFLUENZA LYON — PERIODIC MODEL\n')
        f.write('=' * 80 + '\n\n')
        for row in rows:
            f.write(f"{row['metric']:<50}{row['value']}\n")

def main():
    print('\nLoading periodic-model results...')
    fit = load_fit()
    mle = load_mle()
    posterior = load_posterior()
    performance = load_posterior_performance()
    diagnostics = load_mcmc_diagnostics()
    if len(fit) != 13:
        print(f'WARNING: Expected 13 observations, found {len(fit)}.')
    trajectory_metrics = calculate_trajectory_metrics(fit)
    print_results(fit, mle, posterior, performance, diagnostics, trajectory_metrics)
    print('\nGenerating publication figures...')
    plot_main_trajectory(fit)
    plot_parameter_posteriors(posterior)
    plot_parameter_correlations(posterior)
    plot_performance(performance)
    plot_acceptance(diagnostics)
    plot_rhat(diagnostics)
    plot_residuals(fit)
    plot_observed_vs_fitted(fit)
    save_summary(fit, mle, posterior, performance, diagnostics, trajectory_metrics)
    print('\n' + '=' * 90)
    print('ANALYSIS COMPLETED')
    print('=' * 90)
    print(f'\nOutput directory: {OUT_DIR.resolve()}\n')
    for path in sorted(OUT_DIR.iterdir()):
        print(' ', path.name)
    print('\n')
if __name__ == '__main__':
    main()


In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
FIT_FILE = Path('Influenza_Fit_Results.csv')
OUTPUT_FILE = 'Influenza_Bayesian_Posterior_Lyon.png'

def load_fit():
    """Read the influenza fit file and return the required data arrays."""
    if not FIT_FILE.exists():
        raise FileNotFoundError(f'Required file not found: {FIT_FILE.resolve()}')
    expected_columns = ['week', 'observed', 'mle', 'posterior_median', 'posterior_lower', 'posterior_upper']
    number_pattern = re.compile('[-+]?(?:\\d+\\.?\\d*|\\.\\d+)(?:[EeDd][-+]?\\d+)?')
    rows = []
    with FIT_FILE.open('r', encoding='utf-8-sig') as file:
        for line_number, raw_line in enumerate(file, start=1):
            line = raw_line.strip()
            if not line or line.startswith('#'):
                continue
            if not number_pattern.search(line):
                continue
            values = number_pattern.findall(line)
            if not values:
                continue
            values = [float(value.replace('D', 'E').replace('d', 'e')) for value in values]
            if len(values) != 6:
                if any((char.isalpha() for char in line)):
                    continue
                raise ValueError(f'Line {line_number} contains {len(values)} numerical values; expected 6.\nLine: {raw_line.rstrip()}')
            rows.append(values)
    if not rows:
        raise ValueError(f'No valid six-column data rows found in {FIT_FILE}.')
    df = pd.DataFrame(rows, columns=expected_columns)
    df = df.sort_values('week').reset_index(drop=True)
    if not np.isfinite(df.to_numpy(dtype=float)).all():
        raise ValueError('The fit file contains NaN or infinite values.')
    if (df['posterior_lower'] > df['posterior_upper']).any():
        raise ValueError('Some lower credible limits exceed their upper limits. Check the column order in the source file.')
    return {'week': df['week'].to_numpy(dtype=float), 'r_data': df['observed'].to_numpy(dtype=float), 'mle_curve': df['mle'].to_numpy(dtype=float), 'posterior_median': df['posterior_median'].to_numpy(dtype=float), 'posterior_lower': df['posterior_lower'].to_numpy(dtype=float), 'posterior_upper': df['posterior_upper'].to_numpy(dtype=float)}
fit = load_fit()
weeks_data = fit['week']
r_data = fit['r_data']
mle_curve = fit['mle_curve']
post_median = fit['posterior_median']
lower_ci = fit['posterior_lower']
upper_ci = fit['posterior_upper']
print(f'Loaded {len(weeks_data)} data points from {FIT_FILE}.')
print(f'Week range: {weeks_data.min():g} to {weeks_data.max():g}')
plt.figure(figsize=(11, 7))
plt.fill_between(weeks_data, lower_ci, upper_ci, color='red', alpha=0.25, label='95% Credible Interval')
plt.plot(weeks_data, r_data, 'o:', color='darkgreen', linewidth=2, markersize=7, label='Observed Influenza Data')
plt.plot(weeks_data, post_median, '--', color='black', linewidth=2.5, label='Posterior Median')
ax = plt.gca()
ax.axhline(1, linestyle=':', color='black', linewidth=1.0, label='R(t)=1')
tick_positions = [16, 18, 20, 22, 24, 26, 28]
tick_labels = ['50', '52', '2', '4', '6', '8', '10']
plt.xticks(tick_positions, tick_labels, fontsize=15)
plt.xlim(15.5, 28.5)
ymin = np.floor(min(lower_ci.min(), r_data.min()) * 5) / 5
ymax = np.ceil(max(upper_ci.max(), r_data.max()) * 5) / 5
plt.yticks(np.arange(ymin, ymax + 0.2, 0.4), fontsize=15)
plt.xlabel('Weeks', fontsize=18)
plt.ylabel('Effective Reproduction Number $R(t)$', fontsize=18)
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
ax.tick_params(axis='x', labelsize=17, width=2.5, length=7)
ax.tick_params(axis='y', labelsize=17, width=2.5, length=7)
plt.grid(alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_FILE, dpi=300, bbox_inches='tight')
plt.show()
print(f'Figure saved to: {OUTPUT_FILE}')


In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
FIT_FILE = Path('Sars-Cov2_lyon_MLE_Fit_Results.csv')
MLE_FILE = Path('Sars-Cov2_lyon_MLE_Results.txt')
PERFORMANCE_FILE = Path('Sars-Cov2_lyon_MLE_Posterior_Performance.txt')
POSTERIOR_FILE = Path('Sars-Cov2_lyon_MLE_Posterior_Samples.dat')
MCMC_FILE = Path('Sars-Cov2_lyon_MCMC_Diagnostics_ADDED.dat')
PARAMETER_NAMES = ['alpha', 'season_amp', 'sigma', 'shift_weeks']
SEPARATOR = '-' * 90

def read_text(path):
    if not path.exists():
        return ''
    return path.read_text(encoding='utf-8', errors='ignore')

def extract_numeric_values(text):
    """Extract floating-point numbers, including Fortran D exponents."""
    pattern = '[-+]?(?:\\d+\\.?\\d*|\\.\\d+)(?:[EeDd][-+]?\\d+)?'
    values = re.findall(pattern, text)
    result = []
    for value in values:
        try:
            result.append(float(value.replace('D', 'E').replace('d', 'e')))
        except ValueError:
            pass
    return result

def read_numeric_table(path):
    """
    Read numeric rows from a whitespace- or comma-separated file.
    Handles Fortran scientific notation and comment lines.
    """
    if not path.exists():
        return None
    rows = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as file:
        for line in file:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            line = line.replace('D', 'E').replace('d', 'e')
            line = line.replace(',', ' ')
            values = np.fromstring(line, sep=' ')
            if values.size > 0:
                rows.append(values)
    if not rows:
        return None
    widths = [len(row) for row in rows]
    ncols = max(set(widths), key=widths.count)
    rows = [row for row in rows if len(row) == ncols]
    if not rows:
        return None
    return np.vstack(rows)

def find_labeled_value(text, aliases):
    """
    Find a scalar value associated with a label in a text file.
    Supports forms such as:
        alpha = 0.5
        alpha: 0.5
        alpha    0.5
    """
    for alias in aliases:
        pattern = '(?im)^\\s*' + re.escape(alias) + '\\s*(?:=|:|\\s)\\s*' + '([-+]?(?:\\d+\\.?\\d*|\\.\\d+)(?:[EeDd][-+]?\\d+)?)'
        match = re.search(pattern, text)
        if match:
            try:
                return float(match.group(1).replace('D', 'E').replace('d', 'e'))
            except ValueError:
                pass
    return None

def find_labeled_interval(text, aliases):
    """Extract a two-sided interval, for example [0.2, 0.8]."""
    for alias in aliases:
        pattern = '(?im)^\\s*' + re.escape(alias) + '\\s*(?:=|:)\\s*[\\[\\(]\\s*([-+]?(?:\\d+\\.?\\d*|\\.\\d+)(?:[EeDd][-+]?\\d+)?)\\s*[,;]\\s*([-+]?(?:\\d+\\.?\\d*|\\.\\d+)(?:[EeDd][-+]?\\d+)?)\\s*[\\]\\)]'
        match = re.search(pattern, text)
        if match:
            a = float(match.group(1).replace('D', 'E').replace('d', 'e'))
            b = float(match.group(2).replace('D', 'E').replace('d', 'e'))
            return (min(a, b), max(a, b))
    return None

def print_value(label, value, precision=10):
    if value is None or not np.isfinite(value):
        print(f'{label:<30} = Not available')
    else:
        print(f'{label:<30} = {value:.{precision}f}')

def summarize_array(values):
    """Return count, finite count, non-finite count, min, max and std."""
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    result = {'total': values.size, 'finite': finite.size, 'nonfinite': values.size - finite.size, 'minimum': np.min(finite) if finite.size else np.nan, 'maximum': np.max(finite) if finite.size else np.nan, 'std': np.std(finite) if finite.size else np.nan}
    return result

def print_array_diagnostics(name, values):
    s = summarize_array(values)
    print(f'\n{name}:')
    print(f"  Total values       = {s['total']}")
    print(f"  Finite values      = {s['finite']}")
    print(f"  Non-finite values  = {s['nonfinite']}")
    if s['finite'] > 0:
        print(f"  Minimum            = {s['minimum']:.10f}")
        print(f"  Maximum            = {s['maximum']:.10f}")
        print(f"  Standard deviation = {s['std']:.10f}")

def calculate_metrics(predicted, observed):
    """Calculate Pearson correlation and RMSE using finite pairs only."""
    predicted = np.asarray(predicted, dtype=float)
    observed = np.asarray(observed, dtype=float)
    valid = np.isfinite(predicted) & np.isfinite(observed)
    p = predicted[valid]
    y = observed[valid]
    if len(y) == 0:
        return (np.nan, np.nan, 0)
    rmse = np.sqrt(np.mean((p - y) ** 2))
    if len(y) < 2 or np.std(p) == 0 or np.std(y) == 0:
        pearson_r = np.nan
    else:
        pearson_r = np.corrcoef(p, y)[0, 1]
    return (pearson_r, rmse, len(y))
if not FIT_FILE.exists():
    raise FileNotFoundError(f'Trajectory file not found: {FIT_FILE.resolve()}')
fit_df = pd.read_csv(FIT_FILE)
required_columns = ['week', 'r_data', 'mle_curve', 'posterior_median', 'posterior_lower', 'posterior_upper']
missing = [c for c in required_columns if c not in fit_df.columns]
if missing:
    raise ValueError(f'Missing columns in {FIT_FILE.name}: {missing}\nAvailable columns: {list(fit_df.columns)}')
for column in required_columns:
    fit_df[column] = pd.to_numeric(fit_df[column], errors='coerce')
observed = fit_df['r_data'].to_numpy(dtype=float)
mle_curve = fit_df['mle_curve'].to_numpy(dtype=float)
posterior_median = fit_df['posterior_median'].to_numpy(dtype=float)
posterior_lower = fit_df['posterior_lower'].to_numpy(dtype=float)
posterior_upper = fit_df['posterior_upper'].to_numpy(dtype=float)
n_observations = int(np.isfinite(observed).sum())
mle_r, mle_rmse, mle_n = calculate_metrics(mle_curve, observed)
post_r, post_rmse, post_n = calculate_metrics(posterior_median, observed)
valid_ci = np.isfinite(observed) & np.isfinite(posterior_lower) & np.isfinite(posterior_upper)
if np.any(valid_ci):
    points_inside = (observed[valid_ci] >= posterior_lower[valid_ci]) & (observed[valid_ci] <= posterior_upper[valid_ci])
    n_inside = int(points_inside.sum())
    n_valid_ci = int(valid_ci.sum())
    coverage = 100.0 * n_inside / n_valid_ci
    mean_interval_width = float(np.mean(posterior_upper[valid_ci] - posterior_lower[valid_ci]))
else:
    n_inside = 0
    n_valid_ci = 0
    coverage = np.nan
    mean_interval_width = np.nan
posterior_samples = read_numeric_table(POSTERIOR_FILE)
parameter_samples = None
valid_posterior_samples = 0
number_of_parameters = len(PARAMETER_NAMES)
if posterior_samples is not None:
    if posterior_samples.shape[1] >= number_of_parameters:
        parameter_samples = posterior_samples[:, -number_of_parameters:]
        valid_rows = np.all(np.isfinite(parameter_samples), axis=1)
        parameter_samples = parameter_samples[valid_rows]
        valid_posterior_samples = parameter_samples.shape[0]
mle_text = read_text(MLE_FILE)
performance_text = read_text(PERFORMANCE_FILE)
mcmc_text = read_text(MCMC_FILE)
mle_aliases = {'alpha': ['alpha'], 'season_amp': ['season_amp', 'season amplitude', 'season_ampitude'], 'sigma': ['sigma'], 'shift_weeks': ['shift_weeks', 'shift weeks'], 'logLik': ['logLik', 'log_likelihood', 'log likelihood', 'loglik'], 'RMSE': ['RMSE', 'MLE RMSE'], 'Pearson_r': ['Pearson_r', 'Pearson r', 'MLE Pearson r']}
mle_values = {key: find_labeled_value(mle_text, aliases) for key, aliases in mle_aliases.items()}
if mle_values['RMSE'] is None:
    mle_values['RMSE'] = mle_rmse
if mle_values['Pearson_r'] is None:
    mle_values['Pearson_r'] = mle_r
performance_values = {'r_median': find_labeled_value(performance_text, ['Pearson r median', 'Pearson_r median', 'Pearson median', 'r median']), 'rmse_median': find_labeled_value(performance_text, ['RMSE median', 'RMSE_median', 'posterior RMSE median']), 'r_interval': find_labeled_interval(performance_text, ['Pearson r 95% CrI', 'Pearson_r 95% CrI', 'Pearson r 95% CI', 'Pearson_r_CI']), 'rmse_interval': find_labeled_interval(performance_text, ['RMSE 95% CrI', 'RMSE 95% CI', 'RMSE_CI'])}
performance_table = read_numeric_table(PERFORMANCE_FILE)
if performance_table is not None and performance_table.shape[0] > 1:
    if performance_table.shape[1] >= 2:
        col_r = performance_table[:, -2]
        col_rmse = performance_table[:, -1]
        finite_r = col_r[np.isfinite(col_r)]
        finite_rmse = col_rmse[np.isfinite(col_rmse)]
        if finite_r.size > 1:
            performance_values['r_median'] = float(np.median(finite_r))
            performance_values['r_interval'] = (float(np.percentile(finite_r, 2.5)), float(np.percentile(finite_r, 97.5)))
        if finite_rmse.size > 1:
            performance_values['rmse_median'] = float(np.median(finite_rmse))
            performance_values['rmse_interval'] = (float(np.percentile(finite_rmse, 2.5)), float(np.percentile(finite_rmse, 97.5)))
mcmc_values = {'acceptance': find_labeled_value(mcmc_text, ['overall_acceptance', 'overall acceptance', 'acceptance']), 'posterior_samples': find_labeled_value(mcmc_text, ['posterior_samples', 'posterior samples', 'valid posterior samples']), 'max_rhat': find_labeled_value(mcmc_text, ['max_rhat', 'maximum R-hat', 'maximum Rhat', 'max R-hat']), 'convergence': None}
convergence_match = re.search('(?im)^\\s*(?:convergence(?:_flag)?|convergence flag)\\s*(?:=|:|\\s)\\s*(true|false|\\.true\\.|\\.false\\.|[+-]?\\d+(?:\\.\\d+)?)', mcmc_text)
if convergence_match:
    mcmc_values['convergence'] = convergence_match.group(1)
print('\n' + '=' * 75)
print('POSTERIOR SAMPLE DIAGNOSTICS')
print(SEPARATOR)
if parameter_samples is not None:
    print(f'Valid posterior samples: {valid_posterior_samples}')
    print(f'Number of parameters: {number_of_parameters}')
else:
    print('Valid posterior samples: Not available')
    print(f'Number of parameters expected: {number_of_parameters}')
    print(f'Posterior file checked: {POSTERIOR_FILE.name}')
print('\n' + '=' * 75)
print('TRAJECTORY DATA DIAGNOSTICS')
print(SEPARATOR)
print_array_diagnostics('Observed', observed)
print_array_diagnostics('MLE trajectory', mle_curve)
print_array_diagnostics('Posterior median', posterior_median)
print('\nValid observation-fit pairs:')
print(f'  MLE: {mle_n}')
print(f'  Posterior median: {post_n}')
print('\n' + '=' * 90)
print('Sars-Cov2 LYON — PERIODIC MODEL')
print('=' * 90)
print(f'Number of observations: {n_observations}')
print('\n1. MAXIMUM-LIKELIHOOD ESTIMATE')
print(SEPARATOR)
for parameter in PARAMETER_NAMES:
    print_value(parameter, mle_values.get(parameter))
print_value('logLik', mle_values.get('logLik'))
print_value('RMSE', mle_values.get('RMSE'))
print_value('Pearson_r', mle_values.get('Pearson_r'))
print('\n2. POSTERIOR PARAMETER ESTIMATES')
print(SEPARATOR)
print(f"{'Parameter':<20}{'Median':>16}{'2.5%':>16}{'97.5%':>16}")
if parameter_samples is not None and valid_posterior_samples > 0:
    for j, name in enumerate(PARAMETER_NAMES):
        values = parameter_samples[:, j]
        values = values[np.isfinite(values)]
        if values.size:
            median = np.median(values)
            lower = np.percentile(values, 2.5)
            upper = np.percentile(values, 97.5)
            print(f'{name:<20}{median:>16.8f}{lower:>16.8f}{upper:>16.8f}')
        else:
            print(f"{name:<20}{'Not available':>16}")
else:
    print('Posterior parameter samples could not be read.')
print('\n3. TRAJECTORY FIT')
print(SEPARATOR)
print(f'MLE Pearson r               = {mle_r:.8f}')
print(f'MLE RMSE                    = {mle_rmse:.8f}')
print(f'Posterior median Pearson r  = {post_r:.8f}')
print(f'Posterior median RMSE       = {post_rmse:.8f}')
print('\n4. POSTERIOR PERFORMANCE DISTRIBUTIONS')
print(SEPARATOR)
print_value('Pearson r median', performance_values['r_median'], precision=8)
r_interval = performance_values['r_interval']
if r_interval is not None:
    print(f'Pearson r 95% CrI = [{r_interval[0]:.8f}, {r_interval[1]:.8f}]')
else:
    print('Pearson r 95% CrI = Not available')
print_value('RMSE median', performance_values['rmse_median'], precision=8)
rmse_interval = performance_values['rmse_interval']
if rmse_interval is not None:
    print(f'RMSE 95% CrI = [{rmse_interval[0]:.8f}, {rmse_interval[1]:.8f}]')
else:
    print('RMSE 95% CrI = Not available')
print('\n5. POSTERIOR TRAJECTORY COVERAGE')
print(SEPARATOR)
print(f'Valid observed points = {n_valid_ci}/{len(observed)}')
print(f'Points inside 95% CrI = {n_inside}/{n_valid_ci}')
if np.isfinite(coverage):
    print(f'Coverage = {coverage:.2f}%')
else:
    print('Coverage = Not available')
print_value('Mean 95% interval width', mean_interval_width, precision=8)
print('\n6. MCMC DIAGNOSTICS')
print(SEPARATOR)
print_value('Overall acceptance', mcmc_values['acceptance'], precision=8)
if mcmc_values['posterior_samples'] is not None:
    print(f"Posterior samples = {mcmc_values['posterior_samples']:g}")
elif valid_posterior_samples:
    print(f'Posterior samples = {valid_posterior_samples}')
else:
    print('Posterior samples = Not available')
print_value('Maximum R-hat', mcmc_values['max_rhat'], precision=8)
if mcmc_values['convergence'] is not None:
    print(f"Convergence flag = {mcmc_values['convergence']}")
else:
    print('Convergence flag = Not available')
missing_summary = []
if not mle_text:
    missing_summary.append(MLE_FILE.name)
if not performance_text:
    missing_summary.append(PERFORMANCE_FILE.name)
if not read_text(MCMC_FILE):
    missing_summary.append(MCMC_FILE.name)
if missing_summary:
    print('\n' + '=' * 90)
    print('NOTE: SOME SUMMARY FILES COULD NOT BE READ')
    print('=' * 90)
    for filename in missing_summary:
        print(f'  - {filename}')
    print("\nValues shown as 'Not available' were not inferred or fabricated. Check the relevant file's contents and column labels.")
print('\n' + '=' * 90)
print('END OF Sars-Cov2 LYON RESULTS')
print('=' * 90)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
FIT_FILE = Path('Sars-Cov2_lyon_MLE_Fit_Results.csv')
if not FIT_FILE.exists():
    raise FileNotFoundError(f'Required file not found: {FIT_FILE}')
df = pd.read_csv(FIT_FILE)
required = ['week', 'r_data', 'mle_curve', 'posterior_median', 'posterior_lower', 'posterior_upper']
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}')
weeks = df['week'].to_numpy(float)
observed = df['r_data'].to_numpy(float)
mle = df['mle_curve'].to_numpy(float)
post_median = df['posterior_median'].to_numpy(float)
post_lower = df['posterior_lower'].to_numpy(float)
post_upper = df['posterior_upper'].to_numpy(float)
plt.subplots(figsize=(11, 7))
plt.fill_between(weeks, post_lower, post_upper, color='red', alpha=0.25, label='95% Credible Interval')
plt.plot(weeks, observed, 'o:', color='darkgreen', linewidth=2, markersize=7, label='Sars-Cov-2 data')
plt.plot(weeks, post_median, '--', color='black', linewidth=2.5, label='Posterior Median')
ax = plt.gca()
ax.axhline(1, linestyle=':', color='black', linewidth=1.0, label='R(t)=1')
major_positions = [3, 7, 11, 15, 19, 23, 27]
major_labels = ['37', '41', '45', '49', '1', '5', '9']
ax.set_xticks(major_positions)
ax.set_xticklabels(major_labels, fontsize=15)
ax.set_xlim(2.5, 27.5)
minor_positions = np.arange(3, 28, 2)
ax.set_xticks(minor_positions, minor=True)
ymin = np.floor(min(post_lower.min(), observed.min()) * 5) / 5
ymax = np.ceil(max(post_upper.max(), observed.max()) * 5) / 5
y_ticks = np.arange(ymin, ymax + 0.2, 0.4)
ax.set_yticks(y_ticks)
ax.set_yticklabels([f'{v:.1f}' for v in y_ticks], fontsize=15)
ax.set_xlabel('Weeks', fontsize=18)
ax.set_ylabel('Effective Reproduction Number $R(t)$', fontsize=18)
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
ax.tick_params(axis='x', labelsize=17, width=2.5, length=7)
ax.tick_params(axis='y', labelsize=17, width=2.5, length=7)
ax.grid(which='major', alpha=0.35, linewidth=0.8)
ax.grid(which='minor', alpha=0.2, linewidth=0.5, linestyle='--')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('SARS_CoV2_Bayesian_Posterior_Lyon.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
DATA_DIR = Path('.')
OUTPUT_DIR = DATA_DIR / 'COVID_PUBLICATION_ANALYSIS'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCENARIOS = ['RURAL', 'URBAN']
PARAMETERS = ['alpha', 'season', 'A_week', 'rho', 'sigma']
RHAT_GOOD = 1.01
RHAT_CAUTION = 1.05

def require_file(path):
    if not path.exists():
        raise FileNotFoundError(f'\nRequired file not found:\n  {path.resolve()}')

def optional_file(path):
    return path.exists()

def to_float(x):
    return float(str(x).replace('D', 'E').replace('d', 'e'))

def q95(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return (np.nan, np.nan, np.nan)
    return tuple(np.percentile(x, [2.5, 50.0, 97.5]))

def pearson_r(y, f):
    y = np.asarray(y, dtype=float)
    f = np.asarray(f, dtype=float)
    mask = np.isfinite(y) & np.isfinite(f)
    if np.sum(mask) < 2:
        return np.nan
    if np.std(y[mask]) == 0 or np.std(f[mask]) == 0:
        return np.nan
    return float(np.corrcoef(y[mask], f[mask])[0, 1])

def rmse(y, f):
    y = np.asarray(y, dtype=float)
    f = np.asarray(f, dtype=float)
    mask = np.isfinite(y) & np.isfinite(f)
    if np.sum(mask) == 0:
        return np.nan
    return float(np.sqrt(np.mean((y[mask] - f[mask]) ** 2)))

def read_numeric(path):
    rows = []
    with open(path, 'r', errors='ignore') as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith('#'):
                continue
            parts = s.split()
            try:
                row = [to_float(x) for x in parts]
            except (ValueError, TypeError):
                continue
            rows.append(row)
    if not rows:
        return None
    lengths = {}
    for row in rows:
        lengths[len(row)] = lengths.get(len(row), 0) + 1
    ncol = max(lengths, key=lengths.get)
    rows = [row for row in rows if len(row) == ncol]
    return np.asarray(rows, dtype=float)

def files_for(scenario):
    s = scenario.upper()

def load_fit(path):
    require_file(path)
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    required = ['week', 'r_data', 'mle_curve', 'posterior_median', 'posterior_lower', 'posterior_upper']
    if all((x in df.columns for x in required)):
        try:
            result = {'week': pd.to_numeric(df['week'], errors='raise').to_numpy(float), 'r_data': pd.to_numeric(df['r_data'], errors='raise').to_numpy(float), 'mle_curve': pd.to_numeric(df['mle_curve'], errors='raise').to_numpy(float), 'posterior_median': pd.to_numeric(df['posterior_median'], errors='raise').to_numpy(float), 'posterior_lower': pd.to_numeric(df['posterior_lower'], errors='raise').to_numpy(float), 'posterior_upper': pd.to_numeric(df['posterior_upper'], errors='raise').to_numpy(float)}
            print('\nFIT FILE')
            print('-' * 80)
            print(f'File: {path}')
            print('Format detected: standard comma-separated CSV')
            print(f"Successfully loaded {len(result['week'])} observations.")
            return result
        except (ValueError, TypeError):
            pass
    number_pattern = re.compile('[-+]?(?:\\d+(?:\\.\\d*)?|\\.\\d+)(?:[EeDd][-+]?\\d+)?')
    numeric_rows = []
    with open(path, 'r', errors='ignore') as f:
        for line_no, line in enumerate(f, start=1):
            s = line.strip()
            if not s:
                continue
            if line_no == 1:
                continue
            tokens = number_pattern.findall(s)
            if len(tokens) < 6:
                continue
            try:
                values = [to_float(x) for x in tokens[:6]]
            except (ValueError, TypeError):
                continue
            numeric_rows.append(values)
    if not numeric_rows:
        raise ValueError(f'Could not parse numerical trajectory rows from {path}.')
    arr = np.asarray(numeric_rows, dtype=float)
    if arr.shape[1] != 6:
        raise ValueError(f'Expected six trajectory columns in {path}; found {arr.shape[1]}.')
    result = {'week': arr[:, 0], 'r_data': arr[:, 1], 'mle_curve': arr[:, 2], 'posterior_median': arr[:, 3], 'posterior_lower': arr[:, 4], 'posterior_upper': arr[:, 5]}
    print('\nFIT FILE')
    print('-' * 80)
    print(f'File: {path}')
    print('Format detected: Fortran fixed-width / whitespace numerical rows')
    print(f"Successfully loaded {len(result['week'])} observations.")
    return result

def load_posterior(path):
    require_file(path)
    data = read_numeric(path)
    if data is None:
        raise ValueError(f'No posterior samples found in {path}')
    if data.shape[1] < 5:
        raise ValueError(f'{path} contains {data.shape[1]} columns; 5 parameters are required.')
    return data[:, :5]

def load_parameter_summary(path):
    require_file(path)
    rows = []
    with open(path, 'r', errors='ignore') as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith('#'):
                continue
            parts = s.split()
            if len(parts) < 7:
                continue
            name = parts[0]
            if name not in PARAMETERS:
                continue
            try:
                values = [to_float(x) for x in parts[1:7]]
            except (ValueError, TypeError):
                continue
            rows.append({'parameter': name, 'MLE': values[0], 'posterior_mean': values[1], 'posterior_median': values[2], 'CrI_lower': values[3], 'CrI_upper': values[4], 'Rhat': values[5]})
    if not rows:
        raise ValueError(f'Could not parse parameter summary: {path}')
    df = pd.DataFrame(rows)
    df['parameter'] = pd.Categorical(df['parameter'], categories=PARAMETERS, ordered=True)
    return df.sort_values('parameter').reset_index(drop=True)

def load_diagnostics(path):
    require_file(path)
    lines = path.read_text(errors='ignore').splitlines()
    acceptance = []
    rhat = {}
    mode = None
    for line in lines:
        s = line.strip()
        if not s:
            continue
        if s.startswith('#'):
            low = s.lower()
            if 'chain acceptance' in low:
                mode = 'acceptance'
            elif 'parameter rhat' in low:
                mode = 'rhat'
            else:
                mode = None
            continue
        parts = s.replace(':', ' ').split()
        if mode == 'acceptance' and len(parts) >= 2:
            try:
                chain = int(float(parts[0]))
                value = to_float(parts[1])
                acceptance.append((chain, value))
            except (ValueError, TypeError):
                pass
        elif mode == 'rhat' and len(parts) >= 2:
            name = parts[0]
            if name in PARAMETERS:
                try:
                    rhat[name] = to_float(parts[1])
                except (ValueError, TypeError):
                    pass
    acceptance.sort()
    if rhat:
        rhat_array = np.array([rhat[p] for p in PARAMETERS if p in rhat], dtype=float)
        max_rhat = float(np.max(rhat_array))
    else:
        rhat_array = None
        max_rhat = np.nan
    if np.isfinite(max_rhat):
        if max_rhat <= RHAT_GOOD:
            convergence = 'GOOD'
        elif max_rhat <= RHAT_CAUTION:
            convergence = 'CAUTION'
        else:
            convergence = 'NOT CONVERGED'
    else:
        convergence = 'UNAVAILABLE'
    return {'acceptance': acceptance, 'rhat': rhat, 'rhat_array': rhat_array, 'max_rhat': max_rhat, 'convergence': convergence}

def load_fortran_summary(path):
    require_file(path)
    text = path.read_text(errors='ignore')
    result = {}
    scalar_patterns = {k: v for k, v in patterns.items() if k not in ['boot_r_lo', 'boot_rmse_lo']}
    for key, pattern in scalar_patterns.items():
        m = re.search(pattern, text)
        if m:
            result[key] = to_float(m.group(1))
    m = re.search(patterns['boot_r_lo'], text)
    if m:
        result['boot_r_ci'] = (to_float(m.group(1)), to_float(m.group(2)))
    m = re.search(patterns['boot_rmse_lo'], text)
    if m:
        result['boot_rmse_ci'] = (to_float(m.group(1)), to_float(m.group(2)))
    return result

def analyze_scenario(scenario):
    paths = files_for(scenario)
    fit = load_fit(paths['fit'])
    posterior = load_posterior(paths['posterior'])
    parameter_summary = load_parameter_summary(paths['parameter_summary'])
    diagnostics = load_diagnostics(paths['diagnostics'])
    fortran_summary = load_fortran_summary(paths['summary'])
    mle_r = pearson_r(fit['r_data'], fit['mle_curve'])
    mle_rmse = rmse(fit['r_data'], fit['mle_curve'])
    posterior_median_r = pearson_r(fit['r_data'], fit['posterior_median'])
    posterior_median_rmse = rmse(fit['r_data'], fit['posterior_median'])
    r_lo = np.nan
    r_med = posterior_median_r
    r_hi = np.nan
    rmse_lo = np.nan
    rmse_med = posterior_median_rmse
    rmse_hi = np.nan
    post_perf_file = DATA_DIR / f'{scenario}_COVID_Posterior_Corr_Dist.dat'
    if post_perf_file.exists():
        pp = read_numeric(post_perf_file)
        if pp is not None and pp.shape[1] >= 2:
            r_lo, r_med, r_hi = q95(pp[:, 0])
            rmse_lo, rmse_med, rmse_hi = q95(pp[:, 1])
    inside = (fit['r_data'] >= fit['posterior_lower']) & (fit['r_data'] <= fit['posterior_upper'])
    n_inside = int(np.sum(inside))
    n_total = len(fit['r_data'])
    coverage = 100.0 * n_inside / n_total if n_total else np.nan

def print_scenario_results(result):
    scenario = result['scenario']
    fit = result['fit']
    pstats = result['parameter_summary']
    diagnostics = result['diagnostics']
    fs = result['fortran_summary']
    print('\n')
    print('=' * 100)
    print(f'COVID-19 {scenario} - COMPLETE POST-FORTRAN RESULTS')
    print('=' * 100)
    print('\nDATA')
    print('-' * 100)
    print(f"Number of observations: {len(fit['r_data'])}")
    print('\n1. MLE PARAMETERS')
    print('-' * 100)
    for _, row in pstats.iterrows():
        print(f"{row['parameter']:<18}: {row['MLE']:.12g}")
    if 'mle_loglik' in fs:
        print(f"{'Log-likelihood':<18}: {fs['mle_loglik']:.12g}")
    print(f"{'Pearson r':<18}: {result['mle_r']:.10f}")
    print(f"{'RMSE':<18}: {result['mle_rmse']:.10f}")
    if 'mle_shift' in fs:
        print(f"{'Shift (weeks)':<18}: {fs['mle_shift']:.10f}")
    print('\n2. POSTERIOR PARAMETERS — MEDIAN AND 95% CrI')
    print('-' * 100)
    print(f"{'Parameter':<18}{'MLE':>18}{'Median':>18}{'95% CrI lower':>20}{'95% CrI upper':>20}")
    for _, row in pstats.iterrows():
        print(f"{row['parameter']:<18}{row['MLE']:>18.10f}{row['posterior_median']:>18.10f}{row['CrI_lower']:>20.10f}{row['CrI_upper']:>20.10f}")
    print('\n3. R-HAT FOR EVERY PARAMETER')
    print('-' * 100)
    if diagnostics['rhat']:
        for p in PARAMETERS:
            if p in diagnostics['rhat']:
                print(f"{p:<18}: {diagnostics['rhat'][p]:.10f}")
        print(f"\nMaximum R-hat: {diagnostics['max_rhat']:.10f}")
        print(f"Convergence: {diagnostics['convergence']}")
    else:
        print('R-hat information unavailable.')
    print('\n4. MCMC CHAIN ACCEPTANCE FRACTIONS')
    print('-' * 100)
    if diagnostics['acceptance']:
        for chain, value in diagnostics['acceptance']:
            print(f'Chain {chain:<3}: {value:.10f}')
        print(f"\nMean chain acceptance: {np.mean([x[1] for x in diagnostics['acceptance']]):.10f}")
    else:
        print('Acceptance fractions unavailable.')
    print('\n5. POSTERIOR TRAJECTORY PERFORMANCE')
    print('-' * 100)
    print(f"Posterior median trajectory Pearson r: {result['posterior_median_r']:.10f}")
    print(f"Posterior median trajectory RMSE:       {result['posterior_median_rmse']:.10f}")
    rlo, rmed, rhi = result['posterior_r']
    if np.isfinite(rlo):
        print(f'Posterior Pearson r 95% CrI: [{rlo:.10f}, {rhi:.10f}]')
    else:
        print('Posterior Pearson-r distribution: not separately saved by the supplied COVID Fortran output.')
    elo, emed, ehi = result['posterior_rmse']
    if np.isfinite(elo):
        print(f'Posterior RMSE 95% CrI: [{elo:.10f}, {ehi:.10f}]')
    else:
        print('Posterior RMSE distribution: not separately saved by the supplied COVID Fortran output.')
    print('\n6. 95% LATENT CrI COVERAGE')
    print('-' * 100)
    n_inside, n_total, coverage = result['coverage']
    print(f'Observed points inside: {n_inside}/{n_total}')
    print(f'Coverage: {coverage:.2f}%')
    print('Note: this is latent trajectory coverage, not full predictive coverage.')
    print('-' * 100)
    print('\nPearson r:')
    b = boot['Pearson_r']
    print(f"  95% CI = [{b['ci'][0]:.10f}, {b['ci'][1]:.10f}]")
    print('\nRMSE:')
    b = boot['RMSE']
    print(f"  95% CI = [{b['ci'][0]:.10f}, {b['ci'][1]:.10f}]")
    print('\n8. COMPACT PUBLICATION TABLE')
    print('-' * 100)
    print(f"{'Parameter':<15}{'MLE':>16}{'Posterior median':>22}{'95% CrI':>30}{'Rhat':>14}")
    for _, row in pstats.iterrows():
        rhat = diagnostics['rhat'].get(row['parameter'], np.nan)
        print(f"{row['parameter']:<15}{row['MLE']:>16.8f}{row['posterior_median']:>22.8f}[{row['CrI_lower']:.8f}, {row['CrI_upper']:.8f}]{rhat:>14.8f}")
    print('\nPerformance:')
    print(f"  MLE Pearson r       = {result['mle_r']:.8f}")
    print(f"  MLE RMSE            = {result['mle_rmse']:.8f}")
    print(f"  Posterior r         = {result['posterior_median_r']:.8f}")
    print(f"  Posterior RMSE      = {result['posterior_median_rmse']:.8f}")
    print(f'  Latent CrI coverage = {coverage:.2f}%')
    print('\n' + '=' * 100)

def save_summary(results):
    rows = []
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_DIR / 'COVID_Rural_Urban_Full_Results.csv', index=False)
    return df

def main():
    print('\n')
    print('=' * 100)
    print('COVID-19 RURAL vs URBAN - COMPLETE POST-FORTRAN ANALYSIS')
    print('=' * 100)
    print('\nThis Python script only reads existing Fortran output files.')
    results = []
    for scenario in SCENARIOS:
        result = analyze_scenario(scenario)
        results.append(result)
        print_scenario_results(result)
    save_summary(results)
    print('\n')
    print('=' * 100)
    print('ANALYSIS COMPLETE')
    print('=' * 100)
    print(f'\nSummary saved to:\n  {OUTPUT_DIR.resolve()}')
    print('\nMain output:')
    print(f"  {OUTPUT_DIR / 'COVID_Rural_Urban_Full_Results.csv'}")
if __name__ == '__main__':
    main()


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
DATA_DIR = Path('.')
SCENARIOS = ['RURAL', 'URBAN']
PARAMETERS = ['alpha', 'season', 'A_week', 'rho', 'sigma']
SEPARATOR = '-' * 90
HEADER = '=' * 90

def to_float(value):
    """Convert standard or Fortran scientific notation to float."""
    return float(str(value).replace('D', 'E').replace('d', 'e'))

def print_value(label, value, precision=8):
    """Print a numerical value safely."""
    if value is None:
        print(f'{label:<30} = Not available')
        return
    try:
        value = float(value)
    except (TypeError, ValueError):
        print(f'{label:<30} = Not available')
        return
    if not np.isfinite(value):
        print(f'{label:<30} = Not available')
    else:
        print(f'{label:<30} = {value:.{precision}f}')

def read_text(path):
    path = Path(path)
    if not path.exists():
        return ''
    return path.read_text(encoding='utf-8', errors='ignore')

def read_numeric(path):
    """Read whitespace/comma-separated numerical data, ignoring comments."""
    path = Path(path)
    if not path.exists():
        return None
    rows = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith('#'):
                continue
            s = s.replace(',', ' ')
            try:
                row = [to_float(x) for x in s.split()]
            except (ValueError, TypeError):
                continue
            if row:
                rows.append(row)
    if not rows:
        return None
    widths = [len(row) for row in rows]
    ncols = max(set(widths), key=widths.count)
    rows = [row for row in rows if len(row) == ncols]
    if not rows:
        return None
    return np.asarray(rows, dtype=float)

def extract_labeled_value(text, labels):
    """
    Extract a scalar from lines such as:
        MLE Pearson r = 0.7644
        MLE log-likelihood = -12.345
    """
    number = '([-+]?(?:\\d+\\.?\\d*|\\.\\d+)(?:[EeDd][-+]?\\d+)?)'
    for label in labels:
        pattern = '(?im)^\\s*' + re.escape(label) + '\\s*(?:=|:)\\s*' + number
        match = re.search(pattern, text)
        if match:
            try:
                return to_float(match.group(1))
            except (ValueError, TypeError):
                pass
    return None

def extract_labeled_interval(text, labels):
    """
    Extract two interval endpoints from all of these formats:

        Label = [0.70, 0.82]
        Label = (0.70, 0.82)
        Label = 0.70, 0.82
        Label = 0.70 0.82

    The final format is the one written by your Fortran program.
    """
    number = '[-+]?(?:\\d+\\.?\\d*|\\.\\d+)(?:[EeDd][-+]?\\d+)?'
    for label in labels:
        pattern = '(?im)^\\s*' + re.escape(label) + '\\s*(?:=|:)\\s*' + '[\\[(]?\\s*' + '(' + number + ')' + '(?:\\s*[,;]\\s*|\\s+)' + '(' + number + ')' + '\\s*[\\])]?\\s*$'
        match = re.search(pattern, text)
        if match:
            try:
                a = to_float(match.group(1))
                b = to_float(match.group(2))
                if np.isfinite(a) and np.isfinite(b):
                    return (min(a, b), max(a, b))
            except (ValueError, TypeError):
                pass
    return None

def read_summary_values(path):
    """Read numerical statistics from the Fortran summary file."""
    text = read_text(path)

def read_fit(path):
    """Read the six-column Fit_Results.csv written by Fortran."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Required trajectory file not found: {path.resolve()}')
    columns = ['week', 'r_data', 'mle_curve', 'posterior_median', 'posterior_lower', 'posterior_upper']
    try:
        df = pd.read_csv(path, comment='#')
        df.columns = [str(c).strip() for c in df.columns]
        if all((col in df.columns for col in columns)):
            for col in columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            if df[columns].notna().any().all():
                return {col: df[col].to_numpy(dtype=float) for col in columns}
    except Exception:
        pass
    number = '[-+]?(?:\\d+(?:\\.\\d*)?|\\.\\d+)(?:[EeDd][-+]?\\d+)?'
    rows = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            values = re.findall(number, line)
            if len(values) < 6:
                continue
            try:
                rows.append([to_float(x) for x in values[:6]])
            except ValueError:
                continue
    if not rows:
        raise ValueError(f'Could not parse six-column trajectory file: {path}')
    arr = np.asarray(rows, dtype=float)
    return {col: arr[:, i] for i, col in enumerate(columns)}

def load_parameter_summary(path):
    """
    Expected Fortran format:

    # parameter MLE posterior_mean posterior_median CrI_low CrI_high Rhat
    alpha ...
    season ...
    A_week ...
    rho ...
    sigma ...
    """
    path = Path(path)
    if not path.exists():
        return None
    rows = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith('#'):
                continue
            parts = s.split()
            if len(parts) < 7 or parts[0] not in PARAMETERS:
                continue
            try:
                vals = [to_float(x) for x in parts[1:7]]
            except ValueError:
                continue
            rows.append({'parameter': parts[0], 'MLE': vals[0], 'posterior_mean': vals[1], 'posterior_median': vals[2], 'CrI_lower': vals[3], 'CrI_upper': vals[4], 'Rhat': vals[5]})
    if not rows:
        return None
    df = pd.DataFrame(rows)
    return df.set_index('parameter').reindex(PARAMETERS).reset_index().dropna(subset=['parameter'])

def load_mcmc_diagnostics(path):
    """
    Read the chain acceptance fractions and parameter-specific R-hat
    values written by the supplied Fortran program.
    """
    result = {'acceptance': [], 'rhat': {}}
    path = Path(path)
    if not path.exists():
        return result
    mode = None
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            if s.startswith('#'):
                low = s.lower()
                if 'chain acceptance' in low:
                    mode = 'acceptance'
                elif 'parameter rhat' in low:
                    mode = 'rhat'
                else:
                    mode = None
                continue
            parts = s.split()
            if mode == 'acceptance' and len(parts) >= 2:
                try:
                    result['acceptance'].append((int(parts[0]), to_float(parts[1])))
                except (ValueError, TypeError):
                    pass
            elif mode == 'rhat' and len(parts) >= 2:
                name = parts[0]
                if name in PARAMETERS:
                    try:
                        result['rhat'][name] = to_float(parts[1])
                    except (ValueError, TypeError):
                        pass
    result['acceptance'].sort(key=lambda x: x[0])
    return result

def trajectory_metrics(observed, predicted):
    """Calculate Pearson r and RMSE on paired finite values."""
    observed = np.asarray(observed, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    mask = np.isfinite(observed) & np.isfinite(predicted)
    y = observed[mask]
    p = predicted[mask]
    if len(y) == 0:
        return (np.nan, np.nan, 0)
    rmse = float(np.sqrt(np.mean((y - p) ** 2)))
    if len(y) < 2 or np.std(y) == 0 or np.std(p) == 0:
        r = np.nan
    else:
        r = float(np.corrcoef(y, p)[0, 1])
    return (r, rmse, len(y))

def describe_array(name, values):
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    print(f'\n{name}:')
    print(f'  Total values       = {len(values)}')
    print(f'  Finite values      = {len(finite)}')
    print(f'  Non-finite values  = {len(values) - len(finite)}')
    if len(finite):
        print(f'  Minimum            = {np.min(finite):.10f}')
        print(f'  Maximum            = {np.max(finite):.10f}')
        print(f'  Standard deviation = {np.std(finite):.10f}')

def get_files(region):
    prefix = f'{region}_COVID'
    return {'fit': DATA_DIR / f'{prefix}_Fit_Results.csv', 'posterior': DATA_DIR / f'{prefix}_Posterior_Samples.dat', 'diagnostics': DATA_DIR / f'{prefix}_MCMC_Diagnostics.dat', 'parameter_summary': DATA_DIR / f'{prefix}_Parameter_Summary.dat', 'summary': DATA_DIR / f'{prefix}_Summary.txt'}

def analyze_region(region):
    files = get_files(region)
    fit = read_fit(files['fit'])
    observed = fit['r_data']
    mle_curve = fit['mle_curve']
    post_median = fit['posterior_median']
    post_lower = fit['posterior_lower']
    post_upper = fit['posterior_upper']
    posterior = read_numeric(files['posterior'])
    if posterior is not None:
        if posterior.shape[1] != 5:
            print(f"WARNING: {files['posterior'].name} has {posterior.shape[1]} columns; expected five.")
            posterior = None
    param_summary = load_parameter_summary(files['parameter_summary'])
    mcmc = load_mcmc_diagnostics(files['diagnostics'])
    summary = read_summary_values(files['summary'])
    mle_r_calc, mle_rmse_calc, mle_n = trajectory_metrics(observed, mle_curve)
    post_r_calc, post_rmse_calc, post_n = trajectory_metrics(observed, post_median)
    ci_mask = np.isfinite(observed) & np.isfinite(post_lower) & np.isfinite(post_upper)
    n_ci = int(ci_mask.sum())
    if n_ci:
        inside = (observed[ci_mask] >= post_lower[ci_mask]) & (observed[ci_mask] <= post_upper[ci_mask])
        n_inside = int(inside.sum())
        coverage_calc = 100.0 * n_inside / n_ci
        mean_width = float(np.mean(post_upper[ci_mask] - post_lower[ci_mask]))
    else:
        n_inside = 0
        coverage_calc = np.nan
        mean_width = np.nan
    print('\n\n' + HEADER)
    print(f'COVID-19 {region} REGION — PERIODIC MODEL')
    print(HEADER)
    print('\nPOSTERIOR SAMPLE DIAGNOSTICS')
    print(SEPARATOR)
    if posterior is not None:
        valid_rows = np.all(np.isfinite(posterior), axis=1)
        print(f'Valid posterior samples: {int(valid_rows.sum())}')
        print(f'Number of parameters: {posterior.shape[1]}')
    else:
        print('Valid posterior samples: Not available')
        print('Number of parameters: 5 expected')
    print('\nTRAJECTORY DATA DIAGNOSTICS')
    print(SEPARATOR)
    describe_array('Observed', observed)
    describe_array('MLE trajectory', mle_curve)
    describe_array('Posterior median trajectory', post_median)
    print('\nValid observation-fit pairs:')
    print(f'  MLE                = {mle_n}')
    print(f'  Posterior median   = {post_n}')
    print('\n' + HEADER)
    print(f'COVID-19 {region} — PERIODIC MODEL')
    print(HEADER)
    print(f'Number of observations: {int(np.isfinite(observed).sum())}')
    print('\n1. MAXIMUM-LIKELIHOOD ESTIMATE')
    print(SEPARATOR)
    if param_summary is not None:
        for parameter in PARAMETERS:
            row = param_summary[param_summary['parameter'] == parameter]
            if not row.empty:
                print(f"{parameter:<20} = {row.iloc[0]['MLE']:.10f}")
            else:
                print(f'{parameter:<20} = Not available')
    else:
        print(f"Parameter summary unavailable: {files['parameter_summary'].name}")
    print_value('logLik', summary['loglik'])
    mle_rmse = summary['mle_rmse'] if summary['mle_rmse'] is not None else mle_rmse_calc
    mle_r = summary['mle_r'] if summary['mle_r'] is not None else mle_r_calc
    print_value('RMSE', mle_rmse)
    print_value('Pearson_r', mle_r)
    print_value('MLE shift weeks', summary['mle_shift'])
    print('\n2. POSTERIOR PARAMETER ESTIMATES')
    print(SEPARATOR)
    print(f"{'Parameter':<18}{'Median':>16}{'2.5%':>16}{'97.5%':>16}")
    if posterior is not None:
        valid_rows = np.all(np.isfinite(posterior), axis=1)
        valid_posterior = posterior[valid_rows]
        for j, name in enumerate(PARAMETERS):
            values = valid_posterior[:, j]
            if len(values):
                print(f'{name:<18}{np.median(values):>16.8f}{np.percentile(values, 2.5):>16.8f}{np.percentile(values, 97.5):>16.8f}')
            else:
                print(f'{name:<18}No finite samples')
    elif param_summary is not None:
        for _, row in param_summary.iterrows():
            print(f"{row['parameter']:<18}{row['posterior_median']:>16.8f}{row['CrI_lower']:>16.8f}{row['CrI_upper']:>16.8f}")
    else:
        print('Posterior parameter estimates unavailable.')
    print('\n3. TRAJECTORY FIT')
    print(SEPARATOR)
    print_value('MLE Pearson r', mle_r_calc)
    print_value('MLE RMSE', mle_rmse_calc)
    print_value('Posterior median Pearson r', post_r_calc)
    print_value('Posterior median RMSE', post_rmse_calc)
    print('\n4. POSTERIOR PERFORMANCE DISTRIBUTIONS')
    print(SEPARATOR)
    r_median = summary['posterior_r_median']
    r_ci = summary['posterior_r_ci']
    rmse_median = summary['posterior_rmse_median']
    rmse_ci = summary['posterior_rmse_ci']
    print_value('Pearson r median', r_median)
    if r_ci is not None:
        print(f'Pearson r 95% CrI = [{r_ci[0]:.8f}, {r_ci[1]:.8f}]')
    else:
        print('Pearson r 95% CrI = Not available')
    print_value('RMSE median', rmse_median)
    if rmse_ci is not None:
        print(f'RMSE 95% CrI = [{rmse_ci[0]:.8f}, {rmse_ci[1]:.8f}]')
    else:
        print('RMSE 95% CrI = Not available')
    print('\n5. POSTERIOR TRAJECTORY COVERAGE')
    print(SEPARATOR)
    print(f'Valid observed points = {n_ci}/{len(observed)}')
    print(f'Points inside 95% CrI = {n_inside}/{n_ci}')
    if np.isfinite(coverage_calc):
        print(f'Coverage = {coverage_calc:.2f}%')
    else:
        print('Coverage = Not available')
    print_value('Mean 95% interval width', mean_width)
    if summary['coverage'] is not None:
        print_value('Fortran-reported coverage (%)', summary['coverage'], precision=3)
    print('\n6. MCMC DIAGNOSTICS')
    print(SEPARATOR)
    if posterior is not None:
        valid_rows = np.all(np.isfinite(posterior), axis=1)
        print(f'Posterior samples = {int(valid_rows.sum())}')
    else:
        print('Posterior samples = Not available')
    if mcmc['rhat']:
        max_rhat = max(mcmc['rhat'].values())
        print_value('Maximum R-hat', max_rhat)
        print('\nParameter-specific R-hat:')
        for parameter in PARAMETERS:
            if parameter in mcmc['rhat']:
                print(f"  {parameter:<18} = {mcmc['rhat'][parameter]:.8f}")
    else:
        print('Maximum R-hat = Not available')
    if mcmc['acceptance']:
        print('\nChain acceptance:')
        for chain, value in mcmc['acceptance']:
            print(f'  Chain {chain:<4} = {value:.8f}')
    else:
        print('Chain acceptance = Not available')
    print('\n' + HEADER)
    print(f'END OF {region} REGION RESULTS')
    print(HEADER)
for region in SCENARIOS:
    analyze_region(region)
print('\n' + HEADER)
print('RURAL AND URBAN RESULTS PRINTED')
print(HEADER)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def load_fit_file(scenario):
    path = Path(f'{scenario}_COVID_Fit_Results.csv')
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')
    expected = ['week', 'r_data', 'mle_curve', 'posterior_median', 'posterior_lower', 'posterior_upper']
    try:
        df = pd.read_csv(path, skipinitialspace=True)
        if all((c in df.columns for c in expected)):
            for c in expected:
                df[c] = pd.to_numeric(df[c], errors='coerce')
            df = df.dropna(subset=expected)
            if len(df) > 0:
                return {'week': df['week'].to_numpy(float), 'observed': df['r_data'].to_numpy(float), 'mle': df['mle_curve'].to_numpy(float), 'posterior_median': df['posterior_median'].to_numpy(float), 'posterior_lower': df['posterior_lower'].to_numpy(float), 'posterior_upper': df['posterior_upper'].to_numpy(float)}
    except Exception:
        pass

    def clean_float(x):
        return float(str(x).replace('D', 'E').replace('d', 'e'))
    rows = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.lower().startswith('week'):
                continue
            parts = line.replace(',', ' ').split()
            if len(parts) < 6:
                continue
            try:
                vals = [clean_float(p) for p in parts[:6]]
                rows.append(vals)
            except ValueError:
                continue
    if not rows:
        raise ValueError(f'No numeric data found in {path.name}')
    df = pd.DataFrame(rows, columns=expected)
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    if len(df) == 0:
        raise ValueError(f'All rows dropped after cleaning in {path.name}')
    return {'week': df['week'].to_numpy(float), 'observed': df['r_data'].to_numpy(float), 'mle': df['mle_curve'].to_numpy(float), 'posterior_median': df['posterior_median'].to_numpy(float), 'posterior_lower': df['posterior_lower'].to_numpy(float), 'posterior_upper': df['posterior_upper'].to_numpy(float)}
rural = load_fit_file('RURAL')
urban = load_fit_file('URBAN')
SHIFT = rural['week'][0] - 11.0
rural_week_shifted = rural['week'] - SHIFT
urban_week_shifted = urban['week'] - SHIFT
fig, ax = plt.subplots(figsize=(11, 7))
ax.fill_between(rural_week_shifted, rural['posterior_lower'], rural['posterior_upper'], color='blue', alpha=0.2, label='Rural 95% CrI')
ax.plot(rural_week_shifted, rural['posterior_median'], '-', color='blue', linewidth=2.5, label='Rural posterior median')
ax.fill_between(urban_week_shifted, urban['posterior_lower'], urban['posterior_upper'], color='red', alpha=0.2, label='Urban 95% CrI')
ax.plot(urban_week_shifted, urban['posterior_median'], '--', color='red', linewidth=2.5, label='Urban posterior median')
ax.plot(rural_week_shifted, rural['observed'], ':o', color='green', markersize=6, linewidth=1.5, label='Observed Covid-19 data')
tick_positions = [11, 16, 21, 26, 31, 36, 41]
ax.set_xlim(10.5, 41)
ax.set_xticks(tick_positions)
ax.set_xticklabels([str(t) for t in tick_positions], fontsize=15)
all_obs = np.concatenate([rural['observed'], urban['observed']])
all_lower = np.concatenate([rural['posterior_lower'], urban['posterior_lower']])
all_upper = np.concatenate([rural['posterior_upper'], urban['posterior_upper']])
ymin = np.floor(min(all_lower.min(), all_obs.min()) * 5) / 5
ymax = np.ceil(max(all_upper.max(), all_obs.max()) * 5) / 5
y_ticks = np.arange(ymin, ymax + 0.001, 0.4)
ax.set_yticks(y_ticks)
ax.set_yticklabels([f'{v:.1f}' for v in y_ticks], fontsize=15)
ax.set_xlabel('Weeks', fontsize=18)
ax.set_ylabel('Normalized incidence intensity', fontsize=18)
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
ax.tick_params(axis='x', labelsize=17, width=2.5, length=7)
ax.tick_params(axis='y', labelsize=17, width=2.5, length=7)
ax.grid(which='major', alpha=0.35, linewidth=0.8)
ax.grid(which='minor', alpha=0.2, linewidth=0.5, linestyle='--')
ax.legend(fontsize=11, ncol=2, frameon=True)
plt.tight_layout()
plt.savefig('COVID_Rural_Urban_Bayesian_Mozmb.png', dpi=300, bbox_inches='tight')
plt.show()
